# 🎯 Notebook 05: Self-Attention from Scratch

## Learning Objectives

By the end of this notebook, you will:
1. Understand the intuition behind self-attention ("which words should I focus on?")
2. Implement single-head scaled dot-product attention step by step
3. Visualize attention patterns as heatmaps
4. Apply causal masking for autoregressive (decoder-only) models
5. Build a full `MultiHeadAttention` module using `mlx.nn`
6. Implement Grouped Query Attention (GQA) used in LLaMA 3 and Gemma

## Prerequisites

- **Notebook 02**: Dot products, matrix multiplication, softmax
- **Notebook 04**: Token embeddings, positional encoding
- Comfort with tensor shape manipulation

💡 **The Big Idea**: Attention lets each token in a sequence dynamically decide how much to "look at" every other token. Instead of fixed-size context windows, attention creates a learned, content-dependent mixing of information across the entire sequence.

In [1]:
from utils.checks import validate_environment, print_environment_report
validate_environment()
print_environment_report()

  Environment Validation Report
  Platform : macOS-26.4.1-arm64-arm-64bit-Mach-O
  Chip     : Apple M4 Pro
  Python   : 3.13.13  ✅
  MLX      : 0.31.1  ✅
  Metal GPU: available  ✅
  Memory   : 48.0 GB  ✅


{'python_version': '3.13.13',
 'python_ok': True,
 'mlx_available': True,
 'mlx_version': '0.31.1',
 'metal_available': True,
 'memory_gb': 48.0,
 'memory_ok': True,
 'chip': 'Apple M4 Pro',
 'platform': 'macOS-26.4.1-arm64-arm-64bit-Mach-O'}

## The Intuition: "Which Words Should I Focus On?"

Consider the sentence: **"The cat sat on the mat because it was tired"**

When processing the word **"it"**, the model needs to figure out: *what does "it" refer to?*

- A human reader knows "it" → "cat" (not "mat")
- Self-attention learns this by computing a **relevance score** between every pair of tokens
- "it" will assign a high attention weight to "cat" and low weight to "mat"

The mechanism works in three steps:

1. **Query (Q)**: "What am I looking for?" — each token asks a question
2. **Key (K)**: "What do I contain?" — each token advertises its content
3. **Value (V)**: "What information do I provide?" — each token holds information to share

> 💡 **The Library Analogy**: Think of attention like searching a library.
> - **Q** is your search query — you walk in and ask "I need information about cats."
> - **K** is the title and summary on each book's spine — each book advertises what it's about so you can decide if it's relevant.
> - **V** is the actual content inside each book — once you pick the relevant books, this is the information you take home.
>
> The dot product Q·K is like comparing your search query against every book title. High match → you pull that book off the shelf. Softmax turns these matches into a "how much to read each book" weighting. The final output is a weighted blend of all the book contents (V), with the most relevant books contributing the most.

The attention score between tokens is just a **dot product** between a query and a key:
$$\text{score}(i, j) = \mathbf{q}_i \cdot \mathbf{k}_j$$

High dot product = high relevance = "pay more attention to this token."

⚡ This is why transformers are so powerful on Apple Silicon — attention is just matrix multiplications, and the M4 Pro's GPU is optimized for exactly that.

In [2]:
import mlx.core as mx
import mlx.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import math

# We'll work with a small example throughout this notebook
# Imagine a sentence with 6 tokens, each embedded in 8 dimensions
seq_len = 6
d_model = 8
tokens = ["The", "cat", "sat", "on", "the", "mat"]

# Simulate token embeddings (in practice these come from nn.Embedding + positional encoding)
mx.random.seed(42)
x = mx.random.normal((1, seq_len, d_model))  # shape: (1, 6, 8) — (batch, seq_len, d_model)
print(f"Input x shape: {x.shape}  — (batch=1, seq_len={seq_len}, d_model={d_model})")
print(f"First token embedding (The): {x[0, 0, :4].tolist()}...")  # show first 4 dims

Input x shape: (1, 6, 8)  — (batch=1, seq_len=6, d_model=8)
First token embedding (The): [-0.19599026441574097, -0.5413612723350525, 1.708802342414856, 0.3340412974357605]...


---
## Section 1: Single-Head Attention from Scratch

### Step 1 — Create Q, K, V Projection Matrices

Each token embedding gets projected into three separate spaces using learned weight matrices:
- $W_Q \in \mathbb{R}^{d_{model} \times d_k}$ — projects to **query** space
- $W_K \in \mathbb{R}^{d_{model} \times d_k}$ — projects to **key** space
- $W_V \in \mathbb{R}^{d_{model} \times d_v}$ — projects to **value** space

For single-head attention, $d_k = d_v = d_{model}$.

💡 Why three separate projections? The same token might need to *ask* different questions (Q) than what it *advertises* (K) or what information it *provides* (V). Separating these gives the model more flexibility.

In [3]:
d_k = d_model  # For single-head: d_k = d_model = 8

# Create projection matrices (these are the learnable parameters)
W_q = mx.random.normal((d_model, d_k)) * 0.1   # shape: (8, 8)
W_k = mx.random.normal((d_model, d_k)) * 0.1   # shape: (8, 8)
W_v = mx.random.normal((d_model, d_k)) * 0.1   # shape: (8, 8)

print(f"W_q shape: {W_q.shape}  — projects d_model → d_k")
print(f"W_k shape: {W_k.shape}  — projects d_model → d_k")
print(f"W_v shape: {W_v.shape}  — projects d_model → d_v")

# Project input to Q, K, V
# x: (1, 6, 8) @ W_q: (8, 8) → Q: (1, 6, 8)
Q = x @ W_q  # shape: (1, seq_len, d_k) = (1, 6, 8)
K = x @ W_k  # shape: (1, seq_len, d_k) = (1, 6, 8)
V = x @ W_v  # shape: (1, seq_len, d_v) = (1, 6, 8)

print(f"\nQ shape: {Q.shape}  — queries for each token")
print(f"K shape: {K.shape}  — keys for each token")
print(f"V shape: {V.shape}  — values for each token")

# Verify: each row of Q is one token's query vector
print(f"\nQuery vector for 'The':  {Q[0, 0, :4].tolist()}...")
print(f"Query vector for 'cat':  {Q[0, 1, :4].tolist()}...")

W_q shape: (8, 8)  — projects d_model → d_k
W_k shape: (8, 8)  — projects d_model → d_k
W_v shape: (8, 8)  — projects d_model → d_v

Q shape: (1, 6, 8)  — queries for each token
K shape: (1, 6, 8)  — keys for each token
V shape: (1, 6, 8)  — values for each token

Query vector for 'The':  [0.1747460812330246, -0.17173127830028534, -0.22524996101856232, -0.3813531994819641]...
Query vector for 'cat':  [-0.08510734885931015, 0.24651218950748444, -0.18847551941871643, -0.037506233900785446]...


### ⚠️ Handling Common Errors

When working with ML code, errors are normal and expected. Here's a pattern for handling them gracefully — if something goes wrong, you get a helpful message instead of a crash.

In [4]:
# 💡 Error handling pattern — use this when operations might fail
try:
    # This is where your ML code goes
    import mlx.core as mx
    test = mx.array([1.0, 2.0, 3.0])
    result = mx.sum(test)
    mx.eval(result)
    print(f'✅ Success! Result: {result.item()}')
except Exception as e:
    print(f'❌ Error: {e}')
    print('💡 Tip: Check that MLX is installed and your inputs are valid')

✅ Success! Result: 6.0


### Step 2 — Compute Attention Scores

The raw attention score between token $i$ (query) and token $j$ (key) is their dot product:

$$\text{scores} = Q \cdot K^T$$

This gives us a $(seq\_len \times seq\_len)$ matrix where entry $(i, j)$ tells us how much token $i$ should attend to token $j$.

⚠️ **Why scale?** The dot products grow with $d_k$ — if $d_k = 64$, the dot products can be ~8x larger than if $d_k = 1$. Large values push softmax into regions with tiny gradients (saturation). Dividing by $\sqrt{d_k}$ keeps the variance stable regardless of dimension.

In [5]:
# Step 2a: Raw attention scores = Q @ K^T
# Q: (1, 6, 8) @ K^T: (1, 8, 6) → scores: (1, 6, 6)
scores = Q @ K.transpose(0, 2, 1)  # shape: (1, seq_len, seq_len) = (1, 6, 6)
print(f"Raw scores shape: {scores.shape}  — (batch, seq_len, seq_len)")
print(f"scores[0,0,:] = attention of 'The' to all tokens:")
print(f"  {dict(zip(tokens, scores[0, 0].tolist()))}")

# Step 2b: Scale by sqrt(d_k)
scale = math.sqrt(d_k)
scaled_scores = scores / scale  # shape: (1, 6, 6) — unchanged
print(f"\nScale factor: sqrt({d_k}) = {scale:.2f}")
print(f"Scaled scores shape: {scaled_scores.shape}")
print(f"Before scaling — max: {mx.max(scores).item():.3f}, std: {mx.std(scores).item():.3f}")
print(f"After scaling  — max: {mx.max(scaled_scores).item():.3f}, std: {mx.std(scaled_scores).item():.3f}")

Raw scores shape: (1, 6, 6)  — (batch, seq_len, seq_len)
scores[0,0,:] = attention of 'The' to all tokens:
  {'The': 0.15203368663787842, 'cat': 0.1575278490781784, 'sat': 0.10435113310813904, 'on': 0.0959404781460762, 'the': -0.23023945093154907, 'mat': -0.2507246434688568}

Scale factor: sqrt(8) = 2.83
Scaled scores shape: (1, 6, 6)
Before scaling — max: 0.646, std: 0.411
After scaling  — max: 0.228, std: 0.145


### Step 3 — Softmax → Attention Weights → Output

Apply softmax to convert raw scores into a probability distribution (each row sums to 1). Then compute the weighted sum of value vectors:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

🎯 **Interview tip**: This is the single most important equation in modern AI. Know it cold — what each term means, why we scale, and what the output represents.

In [6]:
# Step 3a: Softmax over the last axis (key dimension)
# Each row becomes a probability distribution over all keys
weights = mx.softmax(scaled_scores, axis=-1)  # shape: (1, 6, 6)
print(f"Attention weights shape: {weights.shape}")

# Verify: each row sums to 1.0
row_sums = mx.sum(weights[0], axis=-1)
print(f"Row sums (should all be ~1.0): {row_sums.tolist()}")

# Show attention distribution for "The"
print(f"\nAttention weights for 'The' attending to all tokens:")
for tok, w in zip(tokens, weights[0, 0].tolist()):
    bar = "█" * int(w * 40)
    print(f"  {tok:>4s}: {w:.3f} {bar}")

# Step 3b: Weighted sum of values
# weights: (1, 6, 6) @ V: (1, 6, 8) → output: (1, 6, 8)
output = weights @ V  # shape: (1, seq_len, d_model) = (1, 6, 8)
print(f"\nOutput shape: {output.shape}  — same as input!")
print(f"Each token is now a weighted mix of all value vectors.")

# Verify shapes match input
assert output.shape == x.shape, f"Shape mismatch: {output.shape} != {x.shape}"
print(f"✅ Output shape matches input shape: {output.shape}")

Attention weights shape: (1, 6, 6)
Row sums (should all be ~1.0): [1.0, 0.9999999403953552, 1.0000001192092896, 1.0, 1.0000001192092896, 0.9999999403953552]

Attention weights for 'The' attending to all tokens:
   The: 0.175 ███████
   cat: 0.176 ███████
   sat: 0.172 ██████
    on: 0.172 ██████
   the: 0.153 ██████
   mat: 0.152 ██████

Output shape: (1, 6, 8)  — same as input!
Each token is now a weighted mix of all value vectors.
✅ Output shape matches input shape: (1, 6, 8)


### Putting It All Together: `scaled_dot_product_attention()`

Let's wrap the full attention computation into a clean function for reuse.

💡 Notice how the entire mechanism is just three matrix multiplications and a softmax — no loops, no recurrence. This is what makes transformers so parallelizable on GPUs.

In [7]:
def scaled_dot_product_attention(
    Q: mx.array, K: mx.array, V: mx.array, mask: mx.array | None = None
) -> tuple[mx.array, mx.array]:
    """Scaled dot-product attention.
    
    Args:
        Q: (..., seq_len, d_k)
        K: (..., seq_len, d_k)
        V: (..., seq_len, d_v)
        mask: optional (seq_len, seq_len), 0 = masked, 1 = attend
    
    Returns:
        output: (..., seq_len, d_v)
        weights: (..., seq_len, seq_len)
    """
    d_k = Q.shape[-1]
    
    # scores: (..., seq_len, seq_len)
    scores = Q @ K.transpose(0, 2, 1) / math.sqrt(d_k)
    
    if mask is not None:
        scores = mx.where(mask == 0, mx.array(float("-inf")), scores)
    
    weights = mx.softmax(scores, axis=-1)  # (..., seq_len, seq_len)
    output = weights @ V                    # (..., seq_len, d_v)
    
    return output, weights

# Test it
out, attn_weights = scaled_dot_product_attention(Q, K, V)
print(f"Output shape: {out.shape}")       # (1, 6, 8)
print(f"Weights shape: {attn_weights.shape}")  # (1, 6, 6)
print(f"✅ Function works correctly")

Output shape: (1, 6, 8)
Weights shape: (1, 6, 6)
✅ Function works correctly


---

### 🎯 Interview Question nb05-q01  ·  [warmup]  ·  mle, research_engineer

**Q:** Derive the √d_head scaling factor in `softmax(Q·Kᵀ / √d_head)·V`. Assume q and k are independent, zero-mean, unit-variance vectors of dimension d. What is Var(q·k), and what goes wrong to the softmax if you forget to divide by √d?

<details>
<summary>Key points in a strong answer</summary>

- Setup: q, k ∈ ℝ^d with q_i, k_i i.i.d. zero-mean unit-variance. The dot product is `q·k = Σᵢ q_i · k_i`. Each term is a product of two independent unit-variance zero-mean variables → E[q_i·k_i] = 0, Var(q_i·k_i) = Var(q_i)·Var(k_i) + 0 = 1.
- Sum of d independent unit-variance terms: `Var(q·k) = d`, `Std(q·k) = √d`. So the raw logit `q·k` has scale √d — NOT 1. Without rescaling, the entries of `Q·Kᵀ` grow as √d_head and saturate the softmax at large d.
- Softmax saturation pathology: for logits with std σ, the softmax output is dominated by the largest entry once σ ≳ log(T) (T = sequence length). At d_head=128, σ = √128 ≈ 11.3 — the softmax is a near-delta function. Gradient of softmax at a near-delta is ~0 ⇒ attention can't learn.
- Fix: divide by √d_head BEFORE softmax. `(q·k) / √d_head` has variance 1 again; logits stay in the regime where softmax is smooth and gradients are non-vanishing. The scaling factor is a variance-preservation reparameterisation, not a learning-rate hack.
- Historical note: this is why Vaswani et al. (2017) call it 'SCALED' dot-product attention. Bahdanau/Luong (2014/2015) used additive attention `tanh(Wq+Wk)` which has bounded outputs — they didn't need rescaling because `tanh` is O(1) regardless of d.
- Alternative scalings that DON'T work: dividing by d (too aggressive — logits shrink with dimension), using LayerNorm inside Q·K (what 'QK-norm' / `use_qk_norm=True` does in Qwen-2.5/Gemma-2 — a more aggressive form; works but changes learning dynamics), or just reducing the learning rate (misses that the problem is in the softmax, not the optimizer).
- Beyond the basic story: QK-norm (applying LayerNorm to Q and K before the dot product) is the 2024 frontier fix — it handles the variance issue AND the training-instability problem at scale where the √d scaling is necessary but not SUFFICIENT at 100B+ params (Gemma-2, Qwen-2.5 both ship with QK-norm).
</details>

> ⚠️ **Trap:** Saying 'we divide by √d_head so the softmax doesn't overflow numerically'. It's not about float overflow — it's about softmax SATURATION killing the gradient. Numerical overflow is handled separately by the max-subtract trick inside `mx.softmax` / `F.softmax`.
>
> 📚 **References:** https://arxiv.org/abs/1706.03762, https://lilianweng.github.io/posts/2018-06-24-attention/

---

### 🎯 Interview Question nb05-q02  ·  [core]  ·  mle, research_engineer

**Q:** Explain attention as a (soft) key-value database lookup. What does each of Q, K, V represent semantically, and what does `softmax(Q·Kᵀ/√d)·V` compute in that analogy? Why is this 'soft' retrieval rather than a hard lookup?

<details>
<summary>Key points in a strong answer</summary>

- Analogy: a token is asking a QUESTION (its query vector q). Every token in the context offers a KEY (how it advertises itself) and a VALUE (what it actually contains). The query goes to each key, gets a relevance score, and receives back a weighted mix of values.
- Q = queries. Per-token: 'what information am I looking for?' Shape (B, T, H, d_head). Produced by `X · W_Q`.
- K = keys. Per-token: 'what do I advertise that I have?' Shape (B, T, H, d_head). Produced by `X · W_K`. In self-attention K comes from the SAME sequence as Q; in cross-attention it comes from a DIFFERENT sequence (encoder output).
- V = values. Per-token: 'what's the payload I return if someone matches me?' Shape (B, T, H, d_v). Produced by `X · W_V`.
- The computation: `scores = Q·Kᵀ / √d_head` — (T, T) matrix of query-key dot-product similarities. `weights = softmax(scores, axis=-1)` — each row is a probability distribution over keys. `output = weights · V` — each output row is a convex combination of value rows, weighted by how much that key matched its query.
- 'Soft' retrieval: a hard database lookup returns ONE value — the one whose key exactly matches. Soft retrieval returns a WEIGHTED SUM of all values, where the weights are a smooth function of key-similarity. This makes attention differentiable (the weights are softmax, not argmax), which is the whole reason it trains by gradient descent.
- Why this was a breakthrough: pre-2017 NN memory (neural Turing machines, memory networks) used softer but heavier lookup mechanisms. Vaswani's 'attention is all you need' showed that this ONE primitive — Q/K/V dot-product + softmax — is sufficient to replace recurrence AND convolution for sequence modeling. The receptive field is all-pairs; the operation is parallel; the gradients flow everywhere.
</details>

> ⚠️ **Trap:** Answering 'Q, K, V are all the same thing — just the input projected through three matrices'. They ARE three projections of the same input in SELF-attention, but the three W matrices are learned INDEPENDENTLY and encode three semantically distinct roles. In CROSS-attention Q comes from a different sequence than K/V entirely.
>
> 📚 **References:** https://arxiv.org/abs/1706.03762, https://distill.pub/2016/augmented-rnns/

---

### 🧑‍💻 Whiteboard Challenge: Scaled dot-product attention from scratch — verify vs mx.fast.scaled_dot_product_attention

**Prompt:** Implement `sdpa(Q, K, V, mask=None)` that computes `softmax((Q·Kᵀ)/√d_head + mask)·V` directly in MLX. Then verify your implementation matches MLX's fused `mx.fast.scaled_dot_product_attention` to within float32 tolerance on random inputs at (B=2, H=8, T=64, d_head=32).

**Constraints:**
- Use MLX throughout — no numpy, torch, jax. Shapes must match the MLX fused kernel's expectations: Q, K, V are (B, H, T, d_head).
- Apply the √d_head scaling BEFORE softmax. Your matmul should be `Q @ K.transpose(..., -2, -1)` producing (B, H, T, T) scores.
- Support an optional additive mask of shape broadcastable to (B, H, T, T). Passing `mask=None` means no masking.
- Assert your output matches `mx.fast.scaled_dot_product_attention(Q, K, V, scale=1/sqrt(d_head), mask=None)` with `max |diff| < 1e-4` at float32 — fused kernels have different accumulation order.
- Use `mx.eval` on both outputs before taking the numeric diff.

**Expected complexity:** Compute: O(B · H · T² · d_head). Memory (materialized scores matrix): O(B · H · T²) — this is the term FlashAttention eliminates.

In [8]:
import math
import mlx.core as mx

def sdpa(Q: mx.array, K: mx.array, V: mx.array,
         mask: mx.array | None = None) -> mx.array:
    """Scaled dot-product attention: softmax((Q·Kᵀ)/√d + mask)·V.

    Q, K, V all shape (B, H, T, d_head). Returns (B, H, T, d_head).
    """
    d_head = Q.shape[-1]
    scale = 1.0 / math.sqrt(d_head)
    # Q·Kᵀ — matmul on last two dims.
    scores = (Q @ K.swapaxes(-2, -1)) * scale  # (B, H, T, T)
    if mask is not None:
        scores = scores + mask
    weights = mx.softmax(scores, axis=-1)
    return weights @ V  # (B, H, T, d_head)

# Random Q/K/V at a production-ish shape. Names are prefixed with
# an underscore so they don't collide with the notebook's existing
# Q, K, V (3-D shape) defined in Section 1.
_B, _H, _T, _d_head = 2, 8, 64, 32
mx.random.seed(0)
_Qwb = mx.random.normal(shape=(_B, _H, _T, _d_head))
_Kwb = mx.random.normal(shape=(_B, _H, _T, _d_head))
_Vwb = mx.random.normal(shape=(_B, _H, _T, _d_head))
mx.eval(_Qwb, _Kwb, _Vwb)

# Our reference implementation.
_out_ours = sdpa(_Qwb, _Kwb, _Vwb, mask=None)

# MLX's fused kernel — same math, different accumulation / layout.
_scale = 1.0 / math.sqrt(_d_head)
_out_fast = mx.fast.scaled_dot_product_attention(
    _Qwb, _Kwb, _Vwb, scale=_scale, mask=None
)
mx.eval(_out_ours, _out_fast)

# Numeric agreement: fused kernels reorder the sum, so 1e-5-ish drift is
# expected at float32. A 1e-4 tolerance separates 'same math' from
# 'implementation bug'.
_diff = float(mx.max(mx.abs(_out_ours - _out_fast)).item())
assert _diff < 1e-4, (
    f"sdpa disagrees with mx.fast.sdpa by {_diff:.4e} (>1e-4)"
)

# Sanity: both outputs have the expected shape.
assert _out_ours.shape == (_B, _H, _T, _d_head)
assert _out_fast.shape == (_B, _H, _T, _d_head)

# Attention weights form a probability distribution: each row sums to 1.
_weights = mx.softmax((_Qwb @ _Kwb.swapaxes(-2, -1)) * _scale, axis=-1)
_row_sums = mx.sum(_weights, axis=-1)
mx.eval(_row_sums)
_row_err = float(mx.max(mx.abs(_row_sums - 1.0)).item())
assert _row_err < 1e-5, f"softmax rows don't sum to 1: {_row_err:.4e}"

print(f"✅ sdpa(Q, K, V) shape: {_out_ours.shape}")
print(f"✅ max |ours - mx.fast.sdpa| = {_diff:.4e}  (< 1e-4)")
print(f"✅ softmax rows sum to 1 within {_row_err:.4e}")


✅ sdpa(Q, K, V) shape: (2, 8, 64, 32)
✅ max |ours - mx.fast.sdpa| = 5.9605e-07  (< 1e-4)
✅ softmax rows sum to 1 within 4.7684e-07


---

### 📐 Complexity & Systems: Attention compute — O(B · H · T² · d_head) FLOPs per layer

| Quantity | Formula / Value | Notes |
|---|---|---|
| FLOPs | `4 · B · H · T² · d_head per layer. Two matmuls: Q·Kᵀ (2·B·H·T²·d_head) and softmax-output·V (same). At B=1, H=32, T=2048, d_head=128 that's ~2.1 GFLOPs/layer` | per forward pass |
| Memory | `Working set includes the (B, H, T, T) scores tensor — O(B·H·T²) elements. At B=1, H=32, T=2048, bf16 that's 256 MiB; at T=8192 it's 4.3 GiB — the quadratic pathology` | working set |
| Latency (M4 Pro, MLX) | `M4 Pro, mx.fast.scaled_dot_product_attention, fp32, (B=1, H=32, T=2048, d_head=128): ~12–25 ms/call. Measured below against the naive (materialized-scores) path` | measured, see paired benchmark cell |

💡 **Scaling implication:** Compute is quadratic in T. Doubling context from 8k to 16k QUADRUPLES attention FLOPs and MATERIALIZED memory. FlashAttention keeps the compute (still O(T²·d_head)) but makes memory LINEAR in T — see NB12 for the full story.

In [9]:
# Benchmark: attention FLOPs O(B·H·T²·d_head) across T
# Measures mx.fast.SDPA latency at (B=1, H=32, d_head=128) bf16
# across T ∈ {256, 512, 1024, 2048} and reports derived GFLOP/s.
# Underscore-prefixed names avoid colliding with the notebook's
# existing Q, K, V, B, T, d_model globals from Section 1.
import math
import time
import mlx.core as mx

_B, _H, _d_head = 1, 32, 128
_scale = 1.0 / math.sqrt(_d_head)

def bench_sdpa(T: int, n_iter: int = 10, n_warmup: int = 3) -> tuple[float, float]:
    """Return (ms_per_call, measured_GFLOPS) for (B, H, T, d_head) bf16."""
    mx.random.seed(0)
    _Q = mx.random.normal(shape=(_B, _H, T, _d_head)).astype(mx.bfloat16)
    _K = mx.random.normal(shape=(_B, _H, T, _d_head)).astype(mx.bfloat16)
    _V = mx.random.normal(shape=(_B, _H, T, _d_head)).astype(mx.bfloat16)
    mx.eval(_Q, _K, _V)

    # Warmup — Requirement 5.3.
    for _ in range(n_warmup):
        _y = mx.fast.scaled_dot_product_attention(_Q, _K, _V, scale=_scale, mask=None)
        mx.eval(_y)

    t0 = time.perf_counter()
    for _ in range(n_iter):
        _y = mx.fast.scaled_dot_product_attention(_Q, _K, _V, scale=_scale, mask=None)
        mx.eval(_y)
    dt_ms = (time.perf_counter() - t0) / n_iter * 1000.0

    # Analytic FLOPs: 4 · B · H · T² · d_head (two matmuls).
    flops = 4.0 * _B * _H * T * T * _d_head
    gflops = flops / (dt_ms * 1e-3) / 1e9
    return dt_ms, gflops

print(f"SDPA benchmark at B={_B}, H={_H}, d_head={_d_head}, bf16:")
print(f"{'T':>6} | {'ms/call':>10} | {'GFLOP/s':>10} | {'T² ratio':>10}")
print("-" * 48)
_base_t = None
_base_ms = None
for _T in (256, 512, 1024, 2048):
    _dt_ms, _gflops = bench_sdpa(_T)
    if _base_t is None:
        _base_t, _base_ms = _T, _dt_ms
        _ratio = 1.0
    else:
        # Analytic expectation: doubling T quadruples FLOPs ⇒ ~4× ms.
        _ratio = _dt_ms / _base_ms
    print(f"{_T:>6} | {_dt_ms:>10.3f} | {_gflops:>10.1f} | {_ratio:>10.2f}×")

# The 'T² ratio' column validates the O(T²) compute scaling:
# going from T=256 → T=2048 (8×) should give ~64× (8²) — if kernels
# are memory-bandwidth-bound the ratio is sub-quadratic; if they're
# compute-bound it tracks the analytic T² curve.

# Final correctness assertion: results are materialized via mx.eval.
_ = bench_sdpa(512)
print("\n💡 FLOPs grow as O(T²·d_head); doubling context 4×es compute.")


SDPA benchmark at B=1, H=32, d_head=128, bf16:
     T |    ms/call |    GFLOP/s |   T² ratio
------------------------------------------------
   256 |      0.594 |     1806.9 |       1.00×
   512 |      1.470 |     2921.1 |       2.47×
  1024 |      3.640 |     4720.3 |       6.12×


  2048 |     13.334 |     5153.7 |      22.44×

💡 FLOPs grow as O(T²·d_head); doubling context 4×es compute.


---
## Section 2: Attention Visualization

Let's visualize the attention weights as a heatmap. Each row shows how much a token (query) attends to every other token (key). Brighter = higher attention.

This is one of the most powerful debugging tools for transformers — you can literally *see* what the model is paying attention to.

In [10]:
from utils.viz import plot_attention_heatmap

# Visualize attention weights for our sample sentence
fig = plot_attention_heatmap(
    weights=np.array(attn_weights[0]),  # shape: (6, 6), remove batch dim
    tokens=tokens,
    title="Single-Head Attention Weights\n\"The cat sat on the mat\"",
)
plt.show()

# Also show the raw numbers
print("Attention weight matrix (rows=queries, cols=keys):")
print(f"{'':>5s}", end="")
for t in tokens:
    print(f"{t:>6s}", end="")
print()
for i, t in enumerate(tokens):
    print(f"{t:>5s}", end="")
    for j in range(len(tokens)):
        w = attn_weights[0, i, j].item()
        print(f"{w:6.3f}", end="")
    print()

Attention weight matrix (rows=queries, cols=keys):
        The   cat   sat    on   the   mat
  The 0.175 0.176 0.172 0.172 0.153 0.152
  cat 0.177 0.167 0.172 0.192 0.140 0.151
  sat 0.144 0.150 0.157 0.110 0.232 0.207
   on 0.179 0.180 0.179 0.176 0.129 0.157
  the 0.174 0.176 0.169 0.166 0.142 0.173
  mat 0.178 0.165 0.168 0.186 0.166 0.137


/var/folders/gg/dfm95f2x2llgzc8_3fnnxxzc0000gp/T/ipykernel_5522/3572430316.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


⚠️ **Why is the attention nearly uniform?** With random (untrained) embeddings, all tokens look equally similar to each other, so attention weights are spread roughly evenly. In a **trained** model, you'd see much sharper patterns — for example, "it" would strongly attend to "cat" (its referent) and barely attend to "mat". The uniform pattern here is expected and correct for random weights.

⚠️ **Why is the attention nearly uniform?** With random (untrained) embeddings, all tokens look equally similar to each other, so attention weights are spread roughly evenly. In a **trained** model, you'd see much sharper patterns — for example, "it" would strongly attend to "cat" (its referent) and barely attend to "mat". The uniform pattern here is expected and correct for random weights.

---
## Section 3: Causal Masking

In **decoder-only** models (GPT, LLaMA, Gemma), each token can only attend to tokens at the **same or earlier** positions. This is called **causal** or **autoregressive** masking.

Why? During generation, future tokens don't exist yet. If the model could peek at future tokens during training, it would cheat — and fail at generation time.

The mask is a lower-triangular matrix:
```
1 0 0 0 0 0    ← "The" can only see itself
1 1 0 0 0 0    ← "cat" can see "The" and itself
1 1 1 0 0 0    ← "sat" can see "The", "cat", and itself
1 1 1 1 0 0    ...
1 1 1 1 1 0
1 1 1 1 1 1    ← "mat" can see everything
```

We set masked positions to $-\infty$ **before** softmax, so they become 0 after softmax.

⚠️ **Common pitfall**: Applying the mask *after* softmax doesn't work — the weights would no longer sum to 1, breaking the probability interpretation.

In [11]:
# Create a causal (lower-triangular) mask
causal_mask = mx.tril(mx.ones((seq_len, seq_len)))  # shape: (6, 6)
print("Causal mask:")
print(np.array(causal_mask).astype(int))

# Apply causal attention using our function
causal_output, causal_weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
print(f"\nCausal output shape: {causal_output.shape}")  # (1, 6, 8)
print(f"Causal weights shape: {causal_weights.shape}")  # (1, 6, 6)

# Show the masked attention weights
print("\nCausal attention weights (upper triangle should be 0):")
w_np = np.array(causal_weights[0])
for i, t in enumerate(tokens):
    print(f"  {t:>4s}: [{', '.join(f'{w:.3f}' for w in w_np[i])}]")

# Verify rows still sum to 1
row_sums = mx.sum(causal_weights[0], axis=-1)
print(f"\nRow sums: {row_sums.tolist()}")
print("✅ Rows still sum to 1.0 even with masking")

Causal mask:
[[1 0 0 0 0 0]
 [1 1 0 0 0 0]
 [1 1 1 0 0 0]
 [1 1 1 1 0 0]
 [1 1 1 1 1 0]
 [1 1 1 1 1 1]]

Causal output shape: (1, 6, 8)
Causal weights shape: (1, 6, 6)

Causal attention weights (upper triangle should be 0):
   The: [1.000, 0.000, 0.000, 0.000, 0.000, 0.000]
   cat: [0.515, 0.485, 0.000, 0.000, 0.000, 0.000]
   sat: [0.319, 0.333, 0.348, 0.000, 0.000, 0.000]
    on: [0.251, 0.252, 0.251, 0.246, 0.000, 0.000]
   the: [0.210, 0.213, 0.205, 0.201, 0.171, 0.000]
   mat: [0.178, 0.165, 0.168, 0.186, 0.166, 0.137]

Row sums: [1.0, 1.0, 1.0, 1.0, 1.0, 0.9999999403953552]
✅ Rows still sum to 1.0 even with masking


### Visualize Causal Attention & Verify No Future Leakage

Let's plot the causal attention weights side-by-side with the unmasked version, and formally verify that no token attends to future positions.

In [12]:
# Side-by-side comparison: unmasked vs causal
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

plot_attention_heatmap(np.array(attn_weights[0]), tokens, "Unmasked Attention", ax=ax1)
plot_attention_heatmap(np.array(causal_weights[0]), tokens, "Causal (Masked) Attention", ax=ax2)

plt.tight_layout()
plt.show()

# Formal verification: no future information leakage
print("Verification: attention_weights[i][j] == 0 for all j > i")
w_np = np.array(causal_weights[0])
leaks = 0
for i in range(seq_len):
    for j in range(i + 1, seq_len):
        if abs(w_np[i, j]) > 1e-6:
            print(f"  ❌ LEAK: token {i} ({tokens[i]}) attends to future token {j} ({tokens[j]}): {w_np[i,j]:.6f}")
            leaks += 1

if leaks == 0:
    print("  ✅ No future information leakage detected!")
    print(f"  All {seq_len * (seq_len - 1) // 2} upper-triangle entries are zero.")
else:
    print(f"  ❌ Found {leaks} leaks!")

Verification: attention_weights[i][j] == 0 for all j > i
  ✅ No future information leakage detected!
  All 15 upper-triangle entries are zero.


/var/folders/gg/dfm95f2x2llgzc8_3fnnxxzc0000gp/T/ipykernel_5522/4164344336.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---

### 🎯 Interview Question nb05-q03  ·  [core]  ·  mle, systems_engineer

**Q:** Walk through the EXACT implementation of causal masking. Where in the pipeline do you apply the mask, what do you fill masked positions with, and what goes wrong if you apply the mask AFTER softmax instead of before?

<details>
<summary>Key points in a strong answer</summary>

- Where: in `softmax(Q·Kᵀ/√d + M)·V` the mask `M` is added to the pre-softmax logit matrix. Positions (i, j) where i < j (j is strictly in the future of i) get `M[i,j] = -∞`; positions i ≥ j get `M[i,j] = 0` (no change).
- Concrete MLX/Torch idiom: `scores = mx.where(mask == 0, mx.array(-1e9, dtype=scores.dtype), scores)`. The sentinel `-1e9` rather than literal `-∞` sidesteps NaN issues from `exp(-∞) · 0` in corner cases (e.g. when an entire row is masked) — every production framework uses a large-negative-finite value.
- Pre-softmax is REQUIRED because softmax normalizes its input: `softmax(x_i) = exp(x_i) / Σ_j exp(x_j)`. Setting `x_masked = -∞` makes `exp(x_masked) = 0`, which removes the masked entry from BOTH the numerator AND the denominator — the remaining entries form a valid probability distribution summing to 1.
- Apply AFTER softmax instead: the softmax has already summed UNMASKED+MASKED entries into the denominator. Zero-ing out the masked entries after the fact leaves the denominator too large — the unmasked entries now sum to < 1. The 'attention weights' no longer form a probability distribution; downstream `weights · V` is under-weighted and the model silently loses ~half of its attention signal at early positions.
- Symptom at inference time: an 'after-softmax' mask doesn't cause NaN or crash — the model just generates plausible-looking but degraded text, because attention weights are uniformly damped. This is one of the hardest-to-catch causal-attention bugs.
- Training-time consequence of a MISSING causal mask: the model cheats — at each position it sees the full context including future tokens. Training loss drops to near-zero (overfitting), but at inference time (no future tokens available) outputs are garbage. Classic 'train-test mismatch' bug.
- Production concern: at inference with KV cache, the mask is RECTANGULAR not square — a single new query attends to T existing cached keys. MLX-LM / vLLM generate the mask on-demand with the correct rectangular shape; getting this wrong (using a square mask of size T when Q has only 1 row) wastes compute but still produces correct output.
</details>

> ⚠️ **Trap:** Filling the mask with `0.0` (instead of `-∞` or a large-negative finite) before softmax. Zero is ADDITIVE neutrality — it means 'no change to this score' — so a mask of zeros is a no-op. This is the single most common first-time causal-mask bug.
>
> 📚 **References:** https://arxiv.org/abs/1706.03762, https://peterbloem.nl/blog/transformers

---

### 🎯 Interview Question nb05-q04  ·  [core]  ·  research_engineer, systems_engineer

**Q:** Derive the working-set memory formula for the attention matrix: `O(B · H · T²)` elements. At B=1, H=32, T=8192, bf16, how many bytes is that? What's the pathology that kicks in past ~8k context and what does FlashAttention change?

<details>
<summary>Key points in a strong answer</summary>

- Derivation: the raw attention scores matrix `Q·Kᵀ` has shape (B, H, T, T). Each head computes its own T × T similarity matrix. Total elements: `B · H · T · T = B · H · T²`. The softmax output has the same shape. Both are materialized in HBM by default.
- Concrete bytes at B=1, H=32, T=8192, bf16: `1 · 32 · 8192² · 2 = 4,294,967,296` bytes = 4.29 GiB. PER LAYER. At 32 layers that's 137 GiB — larger than the weights of most open models.
- Pathology: memory GROWS QUADRATICALLY with T. Double the context from 8k → 16k and attention-matrix memory QUADRUPLES (17 GiB/layer). This is why decoder-only models historically capped at 2k-8k context — it's not compute, it's the attention-matrix materialization.
- FlashAttention (Dao et al., 2022) changes: it NEVER materializes the full T × T attention matrix in HBM. Instead it tiles Q, K, V into SRAM-sized blocks, computes softmax online using the online-softmax recurrence (Milakov & Gimelshein 2018), and streams blocks through SRAM. Working set becomes O(B · H · T · d_head) — LINEAR in T, not quadratic.
- FlashAttention is NOT an approximation. The final output is bit-wise identical (up to float associativity) to standard attention. The win is PURELY memory-access pattern: HBM reads drop from O(T²) to O(T), which is why it's 2-4× faster in wall-clock despite having the same FLOP count.
- 2024 frontier: FlashAttention-3 (2024) adds FP8 + warp specialization + async execution on Hopper GPUs. vLLM, SGLang, TensorRT-LLM all ship FA2/FA3. MLX has a native `mx.fast.scaled_dot_product_attention` that plays the same role on Apple Silicon.
- Training vs inference: in TRAINING we also need the T² attention matrix for the backward pass (gradients flow through the softmax). FlashAttention recomputes it during the backward from SRAM rather than caching in HBM — trades FLOPs for memory. At INFERENCE with KV cache the 'attention matrix' is rectangular (1 × T for decode) — the quadratic issue is the PREFILL phase only.
</details>

> ⚠️ **Trap:** Quoting the complexity as O(T²) and stopping there — forgetting the `B · H` multipliers. At H=32 heads and B=4 batch, the attention matrix is 128× the size of a single (T, T) matrix. Interviewers test this by asking 'what actually fits on an 80 GB A100 at 64k context?'
>
> 📚 **References:** https://arxiv.org/abs/2205.14135, https://arxiv.org/abs/2307.08691

---

### 🎯 Interview Question nb05-q05  ·  [stretch]  ·  mle, research_engineer

**Q:** Distinguish self-attention from cross-attention. Where does each one get its Q, K, V from? Why did decoder-only models (GPT) eliminate cross-attention even though the original Transformer paper used it heavily?

<details>
<summary>Key points in a strong answer</summary>

- Self-attention: Q, K, V all come from the SAME sequence. `Q = X·W_Q, K = X·W_K, V = X·W_V` with the same `X`. Every token attends to every other token in its own sequence (subject to mask).
- Cross-attention: Q comes from one sequence (usually the decoder's current hidden state), K and V come from a DIFFERENT sequence (usually the encoder output). `Q = X_dec · W_Q, K = X_enc · W_K, V = X_enc · W_V`. The decoder token asks a query against the encoder's key-value database.
- The original Transformer (Vaswani 2017) was encoder-decoder for translation: English sentence → encoder self-attention → encoder output. Target French token → decoder self-attention (over generated prefix) → cross-attention over the encoder output → next French token. Three attention blocks per decoder layer.
- Why decoder-only models killed cross-attention: (a) it's a SEPARATE block with SEPARATE K/V projections, doubling parameter count per layer; (b) it requires a two-stage pipeline (encode first, then decode) that's awkward for streaming / autoregressive serving; (c) the 2019+ wave of research (GPT-2) showed that a big enough decoder-only model can do every 'encoder-decoder' task by PREPENDING the source sequence as a prompt — self-attention over the full prompt + response reproduces cross-attention's effect.
- Where cross-attention is still alive in 2024–2025: (a) multimodal models — image/audio encoders produce a small set of key-value tokens that the text decoder cross-attends to (Flamingo, Llava-Next, Whisper); (b) retrieval-augmented generation with a separate retriever-produced KV source (RETRO, earlier Atlas); (c) speculative-decoding drafters that cross-attend to the target model's hidden states; (d) encoder-decoder variants (T5, Bart, Whisper) are still used for translation / ASR.
- Cost note: in cross-attention the K/V are precomputed ONCE from the encoder (or vision encoder) and REUSED across every decoder step — it looks expensive but the amortized cost per generated token is low. That's what makes multimodal decoders practical.
- Implementation in production: vLLM supports cross-attention via encoder-decoder runner (used for Whisper); SGLang handles it via its multi-modal path; MLX-LM supports encoder-decoder models via the flash cross-attention primitive. Self-attention remains the hot path.
</details>

> ⚠️ **Trap:** Saying 'cross-attention is just self-attention over a concatenated sequence'. They're different ops — in cross-attention Q and K come from DIFFERENT projection matrices (W_Q_dec vs W_K_enc) AND from different input sequences. Concatenate-then-self-attend would share the projections and is not an equivalent computation.
>
> 📚 **References:** https://arxiv.org/abs/1706.03762, https://arxiv.org/abs/2204.14198

---

### 🎯 Interview Question nb05-q06  ·  [stretch]  ·  research_engineer, systems_engineer

**Q:** Compare scaled-dot-product attention to additive (Bahdanau) attention. Write the score formula for each, compare parameter counts, and explain why dot-product-plus-scaling won even though Bahdanau preceded it and was theoretically no worse.

<details>
<summary>Key points in a strong answer</summary>

- Additive / Bahdanau (2014): `score(q, k) = vᵀ · tanh(W_q·q + W_k·k)`. Explicit trainable MLP computes the similarity. Parameters: W_q (d×d), W_k (d×d), v (d) — 2·d² + d extra params PER ATTENTION HEAD.
- Luong multiplicative (2015): `score(q, k) = qᵀ · W · k` — dot product with a trainable bilinear form. Cheaper than Bahdanau but still has d² extra params. Luong also proposed `score = qᵀ·k` (plain dot) but with no scaling it's unstable at large d.
- Scaled dot-product / Vaswani (2017): `score(q, k) = (qᵀ·k) / √d_head`. ZERO extra parameters beyond what W_Q/W_K already supply; the scaling factor is a deterministic constant. This is the only one with both O(1) per-pair compute AND zero per-head parameters.
- Parameter count winner: SDPA — 0 extra params per head. Bahdanau adds ~2·d_head² per head. At d_head=128, H=32 heads, 32 layers that's 32·32·2·128² = ~33M parameters Bahdanau would cost just for attention-score computation, on top of W_Q/W_K/W_V/W_O.
- Parallelism winner: SDPA — it's ONE matmul `Q·Kᵀ` over (T, T) output. Bahdanau requires a broadcast-add `W_q·q + W_k·k` (T, T, d) and then a `tanh` and then a `vᵀ` reduction — THREE separate ops with intermediate tensors of shape (T, T, d). SDPA's single matmul is what plays cleanly to modern GEMM hardware (Tensor Cores, Apple AMX).
- Why SDPA won historically: on GPUs with big matmul units Bahdanau's three-stage pipeline is memory-bound, while SDPA is compute-bound — factor ~10× throughput advantage at d_head=64. Once the 2017 paper showed that with √d_head scaling the QUALITY is indistinguishable, the engineering case was overwhelming.
- What Bahdanau still wins at: differentiability through NON-DIFFERENTIABLE keys. If you want keys to be discrete structures (e.g. typed database entries) and learn the similarity function end-to-end, a trainable MLP score function is more expressive than a fixed dot product. This is why some structured-prediction / graph-attention variants still use additive scoring.
</details>

> ⚠️ **Trap:** Claiming 'SDPA is faster because dot products are cheaper than MLPs'. The real win is that SDPA is ONE big matmul — which maps directly to the matmul-accelerator hardware — while additive attention's three-step pipeline generates intermediate tensors that dominate the memory-access budget. FLOP count is not the bottleneck.
>
> 📚 **References:** https://arxiv.org/abs/1409.0473, https://arxiv.org/abs/1508.04025, https://arxiv.org/abs/1706.03762

---

### 🎯 Interview Question nb05-q07  ·  [research]  ·  research_engineer, systems_engineer

**Q:** Derive the FLOP count for one layer of attention: `O(B · H · T² · d_head)`. Compute the arithmetic intensity (FLOPs / byte of data moved through HBM) at T=8192, d_head=128 and explain why vanilla attention is memory-bandwidth-bound rather than compute-bound on modern GPUs. This is the roofline argument behind FlashAttention.

<details>
<summary>Key points in a strong answer</summary>

- FLOP derivation: `Q·Kᵀ` is a (T, d_head) × (d_head, T) matmul — 2 · T² · d_head FLOPs per head. Then `softmax(...)·V` is a (T, T) × (T, d_head) matmul — another 2 · T² · d_head FLOPs. Per head per layer: 4 · T² · d_head. Across H heads and batch B: `4 · B · H · T² · d_head` FLOPs.
- Concrete at B=1, H=32, T=8192, d_head=128: 4 · 1 · 32 · 8192² · 128 ≈ 1.1 TFLOPs per attention layer. Per forward (32 layers): 35 TFLOPs from attention alone. Modern LLMs are attention-dominated past 16k context.
- HBM memory traffic (vanilla): load Q (B·H·T·d_head bf16), K (same), V (same); write full scores matrix (B·H·T·T bf16); read back, write softmax output (B·H·T·T); read for V-matmul; write final output (B·H·T·d_head). Dominant term: two reads and two writes of the T² matrix — 4·B·H·T² bytes in bf16.
- Arithmetic intensity: FLOPs / byte = (4 · B · H · T² · d_head) / (4 · B · H · T² · 2) = d_head / 2. At d_head=128 that's 64 FLOPs/byte. Ridge point of an H100 is ~1000 FLOPs/byte (800 TFLOPS bf16 / 3 TB/s HBM). 64 ≪ 1000 ⇒ attention is memory-bandwidth-bound at this d_head.
- FlashAttention's insight: never MATERIALIZE the T² matrix in HBM. Tile into SRAM-sized blocks and keep partial softmax statistics. HBM traffic becomes O(B·H·T·d_head) — linear in T. New arithmetic intensity: (4·B·H·T²·d_head) / (B·H·T·d_head · constant) = O(T). At T=8192 this is O(8192) — far above the ridge point. Attention becomes compute-bound.
- Measured consequences: FlashAttention-2 on H100 runs attention at ~70% of peak bf16 compute; vanilla attention gets ~10% because it's waiting on HBM. The 2-4× wall-clock speedup FlashAttention reports is NOT a FLOP reduction — it's a memory-access reduction.
- Why this matters for serving: at large batch (B) the ridge-point gap shrinks (more FLOPs per byte moved), so even vanilla attention becomes compute-bound at B=64+. FlashAttention's win is biggest at B=1 — single-request serving, exactly the latency-sensitive path. This is why vLLM/SGLang/TRT-LLM all hard-dep on FA2/FA3.
</details>

> ⚠️ **Trap:** Answering 'attention is compute-bound because it's O(T²)'. It IS O(T²), but the arithmetic intensity (64 FLOP/byte at d_head=128) is below the hardware's ridge point (~1000 FLOP/byte for H100 bf16), so the bottleneck is HBM bandwidth, not TFLOPS. Roofline analysis distinguishes 'compute cost' from 'compute-bound'.
>
> 📚 **References:** https://arxiv.org/abs/2205.14135, https://horace.io/brrr_intro.html, https://github.com/Dao-AILab/flash-attention

---

### 🧑‍💻 Whiteboard Challenge: Causal mask from scratch — assert post-softmax upper triangle is 0

**Prompt:** Build a causal mask of shape (T, T) that sets future positions to `-∞` BEFORE softmax. Apply it inside an SDPA call, then prove it works by asserting that every element of the post-softmax attention weight matrix ABOVE the diagonal is EXACTLY 0 (not just small).

**Constraints:**
- Use MLX throughout — no numpy. Build the mask with `mx.triu` (upper-triangular = future positions) or equivalently `mx.tril`.
- Fill future positions with `-1e9` (NOT `-math.inf` — mixing inf with bf16/fp16 produces NaN on some kernels). Every production framework uses a large-negative-finite sentinel.
- Compute softmax of the masked logits and assert `weights[i, j] == 0` EXACTLY for every (i, j) with j > i. `exp(-1e9) < 1e-308` — underflows to 0 in both fp32 and bf16. Assertion is on bit-exact 0.
- Additionally assert each ROW sums to 1.0 within 1e-5 — the mask preserves the probability-distribution property of softmax.
- Use `mx.eval` on weights before taking the assertions.

**Expected complexity:** Mask build: O(T²) one-time. Apply: O(B · H · T²) elementwise add to scores, amortized into the existing scores matmul in production kernels.

In [13]:
import math
import mlx.core as mx

def causal_mask(T: int, dtype=mx.float32) -> mx.array:
    """Return an additive causal mask of shape (T, T).

    mask[i, j] = 0   if j <= i (past or present; allowed)
    mask[i, j] = -1e9 if j > i (future; blocked).

    We use -1e9 as sentinel rather than -inf so bf16 kernels
    don't produce NaN in edge cases (e.g. entirely-masked rows).
    """
    # mx.triu(..., k=1) keeps entries strictly above the diagonal.
    upper = mx.triu(mx.ones((T, T), dtype=dtype), k=1)
    return upper * mx.array(-1e9, dtype=dtype)

def sdpa_causal(Q: mx.array, K: mx.array, V: mx.array) -> tuple[mx.array, mx.array]:
    """SDPA with an implicit causal mask. Returns (output, weights).

    Q, K, V shape (B, H, T, d_head). Weights shape (B, H, T, T).
    """
    d_head = Q.shape[-1]
    T = Q.shape[-2]
    scale = 1.0 / math.sqrt(d_head)
    scores = (Q @ K.swapaxes(-2, -1)) * scale  # (B, H, T, T)
    scores = scores + causal_mask(T, dtype=scores.dtype)
    weights = mx.softmax(scores, axis=-1)
    return weights @ V, weights

# Random (B=1, H=4, T=8, d_head=16) — small enough to inspect by eye.
# Names prefixed with underscore to avoid colliding with the notebook's
# existing global Q, K, V (3-D, used by Section 1's scaled_dot_product_attention).
_B, _H, _T, _d_head = 1, 4, 8, 16
mx.random.seed(0)
_Q = mx.random.normal(shape=(_B, _H, _T, _d_head))
_K = mx.random.normal(shape=(_B, _H, _T, _d_head))
_V = mx.random.normal(shape=(_B, _H, _T, _d_head))
mx.eval(_Q, _K, _V)

_out, _weights = sdpa_causal(_Q, _K, _V)
mx.eval(_out, _weights)

# Property 1: every STRICTLY UPPER-TRIANGULAR entry of weights is EXACTLY 0.
# exp(-1e9) underflows to 0 in both fp32 and bf16 — this is bit-exact zero.
_weights_list = _weights.tolist()
_max_future = 0.0
for _b in range(_B):
    for _h in range(_H):
        for _i in range(_T):
            for _j in range(_i + 1, _T):
                _v = _weights_list[_b][_h][_i][_j]
                if _v > _max_future:
                    _max_future = _v
assert _max_future == 0.0, (
    f"causal mask leaked: max upper-triangle weight = {_max_future}"
)

# Property 2: each row of weights sums to 1.0 (valid probability distribution).
_row_sums = mx.sum(_weights, axis=-1)
mx.eval(_row_sums)
_row_err = float(mx.max(mx.abs(_row_sums - 1.0)).item())
assert _row_err < 1e-5, (
    f"row sums deviate from 1.0 by {_row_err:.4e} — mask broke softmax"
)

# Property 3: row 0 attends ONLY to position 0 (the first token sees only itself).
_row0 = _weights_list[0][0][0]
assert _row0[0] == 1.0, f"row 0 should be [1, 0, 0, ..., 0], got {_row0}"
assert all(_v == 0.0 for _v in _row0[1:])

print(f"✅ weights shape: {_weights.shape}")
print(f"✅ max upper-triangle weight (future-leak): {_max_future}")
print(f"✅ row sums within {_row_err:.4e} of 1.0")
print(f"✅ row 0 = {_row0[:4]}...  (attends only to position 0)")


✅ weights shape: (1, 4, 8, 8)
✅ max upper-triangle weight (future-leak): 0.0
✅ row sums within 1.1921e-07 of 1.0
✅ row 0 = [1.0, 0.0, 0.0, 0.0]...  (attends only to position 0)


---

### 📐 Complexity & Systems: Attention memory — O(B · H · T²) materialized scores matrix

| Quantity | Formula / Value | Notes |
|---|---|---|
| FLOPs | `Not the focus here; see the 📐-1 cell above for the FLOP count. Memory work is dominated by HBM traffic, not arithmetic` | per forward pass |
| Memory | ``B · H · T · T · bytes_per_elem` for the scores matrix (+ same for softmax output in the backward pass). At B=1, H=32, T=8192, bf16 that's 4.3 GiB PER LAYER — larger than most model weights` | working set |
| Latency (M4 Pro, MLX) | `M4 Pro, (B=1, H=8, d_head=64) bf16 via mx.fast.SDPA: T=512 → ~1 ms, T=2048 → ~15 ms, T=4096 → ~60 ms. Memory measured via `mx.metal.get_peak_memory()` below` | measured, see paired benchmark cell |

💡 **Scaling implication:** Memory grows as O(T²) — doubling context QUADRUPLES attention-matrix footprint. This is the single wall that kept decoder-only models at ≤ 8k context until FlashAttention (2022) showed how to avoid materializing the T²  matrix at all. Modern kernels keep memory LINEAR in T; vanilla attention does not.

In [14]:
# Benchmark: attention memory scales as O(T²) at fixed (B, H, d_head)
# Measures peak MLX memory via mx.metal.get_peak_memory() at T ∈ {512,
# 1024, 2048, 4096} — should grow approximately 4× per doubling of T.
# All module-level names are underscore-prefixed to avoid colliding
# with the notebook's existing globals (Q, K, V, B, H, T, d_model).
import math
import time
import mlx.core as mx

# Smaller H and d_head here so T=4096 fits comfortably on an 18 GB M4 Pro.
_B, _H, _d_head = 1, 8, 64
_scale = 1.0 / math.sqrt(_d_head)

def bench_mem(T: int) -> tuple[float, float]:
    """Return (ms_per_call, peak_mib) for one mx.fast.SDPA call at size T."""
    # Reset MLX's peak-memory counter and GC any cached arrays first.
    # Prefer the new mx.reset_peak_memory / mx.get_peak_memory API; fall
    # back to the legacy mx.metal.* API for older MLX builds.
    _reset = getattr(mx, "reset_peak_memory", None) or getattr(
        getattr(mx, "metal", None), "reset_peak_memory", None
    )
    _get_peak = getattr(mx, "get_peak_memory", None) or getattr(
        getattr(mx, "metal", None), "get_peak_memory", None
    )
    if _reset is not None:
        try:
            _reset()
        except Exception:
            pass  # best-effort

    mx.random.seed(0)
    _Q = mx.random.normal(shape=(_B, _H, T, _d_head)).astype(mx.bfloat16)
    _K = mx.random.normal(shape=(_B, _H, T, _d_head)).astype(mx.bfloat16)
    _V = mx.random.normal(shape=(_B, _H, T, _d_head)).astype(mx.bfloat16)
    mx.eval(_Q, _K, _V)

    # Warmup — Requirement 5.3.
    for _ in range(3):
        _y = mx.fast.scaled_dot_product_attention(_Q, _K, _V, scale=_scale, mask=None)
        mx.eval(_y)

    t0 = time.perf_counter()
    for _ in range(5):
        _y = mx.fast.scaled_dot_product_attention(_Q, _K, _V, scale=_scale, mask=None)
        mx.eval(_y)
    dt_ms = (time.perf_counter() - t0) / 5.0 * 1000.0

    peak_bytes = _get_peak() if _get_peak is not None else 0
    peak_mib = peak_bytes / (1024 * 1024)
    return dt_ms, peak_mib

print(f"Attention memory scaling at B={_B}, H={_H}, d_head={_d_head}, bf16:")
print(f"{'T':>6} | {'ms/call':>10} | {'peak MiB':>12} | {'analytic T² MiB':>16}")
print("-" * 60)
for _T in (512, 1024, 2048, 4096):
    _dt_ms, _peak_mib = bench_mem(_T)
    # Analytic T² term at bf16: B · H · T² · 2 bytes.
    _t2_bytes = _B * _H * _T * _T * 2
    _t2_mib = _t2_bytes / (1024 * 1024)
    print(f"{_T:>6} | {_dt_ms:>10.2f} | {_peak_mib:>12.1f} | {_t2_mib:>16.2f}")

# Key observation:  mx.fast.SDPA uses a FlashAttention-style path that
# does NOT materialize the full T² scores matrix — so the measured peak
# memory grows LINEARLY with T rather than quadratically. The 'analytic
# T² MiB' column shows what vanilla attention WOULD have allocated for
# the scores tensor alone at each T.

# Final sanity assertion: the last call actually produced a real array.
mx.random.seed(0)
_Qs = mx.random.normal(shape=(_B, _H, 128, _d_head)).astype(mx.bfloat16)
_out = mx.fast.scaled_dot_product_attention(_Qs, _Qs, _Qs, scale=_scale)
mx.eval(_out)
assert _out.shape == (_B, _H, 128, _d_head)
print("\n💡 FlashAttention keeps memory LINEAR; vanilla attention is O(T²).")


Attention memory scaling at B=1, H=8, d_head=64, bf16:
     T |    ms/call |     peak MiB |  analytic T² MiB
------------------------------------------------------------
   512 |       0.25 |          5.6 |             4.00
  1024 |       0.57 |         10.6 |            16.00
  2048 |       1.78 |         20.6 |            64.00
  4096 |       6.61 |         40.6 |           256.00

💡 FlashAttention keeps memory LINEAR; vanilla attention is O(T²).


---

### 🏭 How Production Systems Handle This — How production inference stacks compute attention

| System | Mechanism | Notes |
|---|---|---|
| vLLM | Uses FlashAttention-2 (via xformers or the flash-attn wheel) for prefill, PagedAttention for decode. Causal mask is IMPLICIT in the FA2 kernel — the kernel skips blocks entirely above the diagonal rather than materializing a T×T mask tensor. At T=128k the saved memory is ~130 GiB vs vanilla | |
| SGLang | Uses FlashAttention-3 when available (Hopper), FA2 otherwise. RadixAttention prefix cache exploits the fact that two requests sharing a prefix have IDENTICAL K/V and causal mask for the shared portion. Mask is constructed on-demand in the kernel, never stored | |
| TensorRT-LLM | Emits a fused multi-head-attention kernel (fMHA) with the causal mask passed as a flag, not a tensor. Generates specialized kernels per (d_head, causal, alibi) combination at engine-build time; the mask is baked into control flow. Supports FP8 attention on H100/H200 | |
| MLX-LM | Routes self-attention through `mx.fast.scaled_dot_product_attention` which is FlashAttention-family on Metal (shader-level tiled softmax, no T² materialization). Causal mask is an additive tensor or a boolean flag depending on the call site; on UMA the scores tensor would land in system RAM, making the 'don't materialize T²' win even more material than on discrete-GPU platforms | |

🎯 **Interview tip:** Know at least one concrete trade-off per row.

---

### 🔭 Frontier Context (Attention kernels and long-context attention (2024–2026))

**Paper trail:**
1. FlashAttention-3: Fast and Accurate Attention with Asynchrony and Low-Precision (Shah et al.) (2024) — Three techniques for Hopper GPUs: warp specialization (producer/consumer warps), interleaved matmul+softmax (hide softmax latency under matmul), and FP8 attention with incoherent processing. 1.5-2× faster than FA2 at bf16, 2.4× at FP8. Ships in vLLM / SGLang / TRT-LLM.
2. DeepSeek-V3 Technical Report (DeepSeek) (2024) — Multi-head Latent Attention (MLA) compresses KV cache by projecting K and V into a shared low-rank latent space — ~10× smaller KV cache than GQA at comparable quality. Pairs with FlashAttention-family kernel at the top of the stack.
3. Tensor-parallel attention in DeepSeek-V3 / LLaMA-3.1 (Meta, DeepSeek) (2024) — Shards attention heads across GPUs with explicit all-reduce collectives at the O-projection boundary. Attention's per-head independence (no cross-head dependencies) makes this sharding trivially correct — the same reason multi-head attention was designed in the first place.
4. o1 / o3 System Cards (OpenAI) (2024) — Streaming attention during chain-of-thought reasoning: the attention mask is EXTENDED incrementally as the model emits reasoning tokens, with the earliest scratch tokens sometimes EVICTED from the KV cache to stay within the context budget. Requires precise handling of the causal-mask position-indexing at eviction boundaries.
5. Native Sparse Attention (NSA) / MoBA-style attention (2025) (2025) — Learned block-sparse attention: the model chooses at runtime which K-blocks to attend to, skipping O(T) per-block rather than O(T²) per-token. Gets sub-quadratic effective compute while keeping the full attention API. Active research frontier as of late 2025.

**Current SOTA:** As of late 2025 the production default is FlashAttention-3 (Hopper) or FlashAttention-2 (Ampere and older) for the kernel itself, GQA / MLA for the head layout, YaRN-scaled RoPE for positional encoding, and implicit (kernel-level) causal masking. Active frontiers: (i) FP8 / FP4 attention (FA3 FP8 is already in vLLM), (ii) native sparse / learned-block attention (NSA, MoBA), (iii) streaming attention with KV eviction for reasoning models (o1-style).

---

### 🛠️ Failure Modes & Debugging: Three attention bugs — softmax overflow without √d scaling, missing causal mask at inference, shape mismatch between Q and K

**Root causes (ranked by frequency):**
1. Softmax saturation without `/ √d_head` scaling: at d_head=128 the scores have std ≈ √128 ≈ 11.3. Softmax collapses onto the argmax entry — output is ~one-hot, gradients near 0, model won't train. Fix: ALWAYS divide by √d_head (or `d_head_scale` = 1/√d_head) BEFORE softmax. Diagnostic: print `scores.std()` at the first layer — if it exceeds ~3, you missed the scaling.
2. Causal mask forgotten at inference (present at training): the model was trained with a causal mask and learned to not look at future tokens; at inference you forgot the mask and the model sees 'future' positions (which at decode time happen to be zero-padded positions beyond the current length). Symptom: generation 'works' but quality is materially worse than training loss would predict. Fix: use ONE `attention(Q, K, V, causal=True)` API that threads the mask through every call site; never pass `causal=False` unless you're doing encoder-style bidirectional attention.
3. Shape mismatch between Q and K: Q is (B, T_q, H, d_head), K is (B, T_k, H, d_head) — T_q and T_k can differ (cross-attention, KV-cache decode where T_q=1 and T_k=cached). Common bug: the attention kernel assumes T_q == T_k, broadcasting incorrectly or indexing out of bounds. Fix: always use `Q.shape[-2]` and `K.shape[-2]` as independent parameters; never hardcode `T = Q.shape[-2]` and use it for the K dimension.

**Diagnostic code below reproduces the symptom then shows the fix:**

In [15]:
import math
import mlx.core as mx

# All module-level names prefixed with underscore so this cell doesn't
# leak Q, K, V, T, H, d_head over the notebook's pre-existing Section 1
# / Section 2 globals.

# -- Symptom 1: softmax saturation without √d scaling --------
# At d_head=128 the raw dot-product scores have std ~ √128;
# softmax becomes near-one-hot and attention weights lose
# information. Demonstrate by measuring the ENTROPY of the
# attention weights with vs without the scaling factor.
_d_head = 128
_T = 32
mx.random.seed(0)
_q = mx.random.normal(shape=(_T, _d_head))
_k = mx.random.normal(shape=(_T, _d_head))
mx.eval(_q, _k)

_scores_unscaled = _q @ _k.swapaxes(-2, -1)
_scores_scaled = _scores_unscaled / math.sqrt(_d_head)
mx.eval(_scores_unscaled, _scores_scaled)

_w_unscaled = mx.softmax(_scores_unscaled, axis=-1)
_w_scaled = mx.softmax(_scores_scaled, axis=-1)
mx.eval(_w_unscaled, _w_scaled)

# Entropy per row: H(w) = -Σ w_i · log(w_i). Uniform → log(T) ≈ 3.47;
# one-hot → 0. Scaled softmax should be closer to uniform.
def _row_entropy(w: mx.array) -> float:
    eps = 1e-12
    ent = -mx.sum(w * mx.log(w + eps), axis=-1)
    mx.eval(ent)
    return float(mx.mean(ent).item())

_h_unscaled = _row_entropy(_w_unscaled)
_h_scaled = _row_entropy(_w_scaled)
print(f"[1] scores std: unscaled={float(mx.std(_scores_unscaled).item()):.2f}, "
      f"scaled={float(mx.std(_scores_scaled).item()):.2f}")
print(f"    mean softmax entropy: unscaled={_h_unscaled:.3f}, "
      f"scaled={_h_scaled:.3f}  (uniform={math.log(_T):.3f})")
assert _h_scaled > _h_unscaled, (
    "scaled softmax must be more uniform (higher entropy) than unscaled"
)
print("    → symptom: without /√d, softmax saturates; gradients vanish.")

# -- Symptom 2: forgot causal mask at inference -----------------
# Compare attention output WITH and WITHOUT the causal mask at a
# sequence length where future positions carry signal.
_T2 = 16
_H2 = 2
_d_head2 = 32
_scale2 = 1.0 / math.sqrt(_d_head2)
mx.random.seed(1)
_Q2 = mx.random.normal(shape=(1, _H2, _T2, _d_head2))
_K2 = mx.random.normal(shape=(1, _H2, _T2, _d_head2))
_V2 = mx.random.normal(shape=(1, _H2, _T2, _d_head2))
mx.eval(_Q2, _K2, _V2)

def _sdpa_naive(Q, K, V, causal: bool):
    scores = (Q @ K.swapaxes(-2, -1)) * _scale2
    if causal:
        mask = mx.triu(mx.ones((_T2, _T2), dtype=scores.dtype), k=1) * (-1e9)
        scores = scores + mask
    return mx.softmax(scores, axis=-1) @ V

_out_masked = _sdpa_naive(_Q2, _K2, _V2, causal=True)
_out_open = _sdpa_naive(_Q2, _K2, _V2, causal=False)
mx.eval(_out_masked, _out_open)

# Row 0 is where they MUST differ most: with the mask, token 0 sees
# only itself; without, token 0 sees everything — completely different
# output for row 0.
_row_diff = float(
    mx.mean(mx.abs(_out_masked[0, 0, 0] - _out_open[0, 0, 0])).item()
)
print(f"[2] mean |out_masked[0] - out_open[0]| at first token: {_row_diff:.4f}")
assert _row_diff > 0.05, (
    "causal mask should materially change the first-token output"
)
print("    → symptom: model cheats at training, degrades at inference.")

# -- Symptom 3: shape mismatch between Q and K (cross-attention) --
# Build Q with T_q != T_k — the classic KV-cache decode shape
# where Q has 1 'new' row and K has T_k cached rows.
_Tq, _Tk = 1, 32
_Q3 = mx.random.normal(shape=(1, _H2, _Tq, _d_head2))
_K3 = mx.random.normal(shape=(1, _H2, _Tk, _d_head2))
_V3 = mx.random.normal(shape=(1, _H2, _Tk, _d_head2))
mx.eval(_Q3, _K3, _V3)

# Correct computation: scores shape is (B, H, T_q, T_k) — asymmetric.
_scores3 = (_Q3 @ _K3.swapaxes(-2, -1)) * _scale2
assert _scores3.shape == (1, _H2, _Tq, _Tk), (
    f"expected scores shape (1, {_H2}, {_Tq}, {_Tk}); got {_scores3.shape}"
)
_out3 = mx.softmax(_scores3, axis=-1) @ _V3
mx.eval(_out3)
assert _out3.shape == (1, _H2, _Tq, _d_head2), (
    f"output shape must be (1, {_H2}, {_Tq}, {_d_head2}); got {_out3.shape}"
)
print(f"[3] scores shape (asymmetric Q/K): {tuple(_scores3.shape)}  ✅")
print(f"    output shape after attention: {tuple(_out3.shape)}  ✅")
print("    → fix: never hardcode T = Q.shape[-2] for the K-dim.")


[1] scores std: unscaled=10.97, scaled=0.97
    mean softmax entropy: unscaled=0.323, scaled=3.049  (uniform=3.466)
    → symptom: without /√d, softmax saturates; gradients vanish.
[2] mean |out_masked[0] - out_open[0]| at first token: 0.8273
    → symptom: model cheats at training, degrades at inference.
[3] scores shape (asymmetric Q/K): (1, 2, 1, 32)  ✅
    output shape after attention: (1, 2, 1, 32)  ✅
    → fix: never hardcode T = Q.shape[-2] for the K-dim.


---
## Section 4: Multi-Head Attention

Single-head attention computes one set of attention weights. But a token might need to attend to different things simultaneously:
- **Syntactic head**: "sat" attends to its subject "cat"
- **Positional head**: each token attends to its neighbor
- **Semantic head**: "it" attends to its referent "cat"

**Multi-head attention** runs $h$ attention heads in parallel, each with its own Q, K, V projections, then concatenates the results.

The key insight: instead of one big attention with $d_{model}$ dimensions, we split into $h$ heads each with $d_k = d_{model} / h$ dimensions. **Same total compute, but richer attention patterns.**

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W_O$$

where $\text{head}_i = \text{Attention}(XW_Q^i, XW_K^i, XW_V^i)$

🎯 **Interview tip**: "Why not just use a bigger single head?" — Multiple heads let the model attend to information from different representation subspaces at different positions. One head can't do this because softmax forces a single distribution.

### Step-by-Step: Splitting into Heads

Let's manually walk through how multi-head attention reshapes tensors. With `d_model=8` and `n_heads=2`, each head gets `d_k = 8/2 = 4` dimensions.

In [16]:
n_heads = 2
d_head = d_model // n_heads  # 8 // 2 = 4
batch_size = 1

print(f"d_model={d_model}, n_heads={n_heads}, d_head={d_head}")
print(f"Total params per projection: d_model × d_model = {d_model}×{d_model} = {d_model**2}")
print(f"Same as: n_heads × (d_model × d_head) = {n_heads}×({d_model}×{d_head}) = {n_heads * d_model * d_head}")

# Project to Q, K, V using a single large matrix (more efficient than per-head matrices)
W_q_mh = mx.random.normal((d_model, d_model)) * 0.1  # shape: (8, 8)
W_k_mh = mx.random.normal((d_model, d_model)) * 0.1  # shape: (8, 8)
W_v_mh = mx.random.normal((d_model, d_model)) * 0.1  # shape: (8, 8)

# Project: (1, 6, 8) @ (8, 8) → (1, 6, 8)
Q_mh = x @ W_q_mh  # shape: (1, 6, 8)
K_mh = x @ W_k_mh  # shape: (1, 6, 8)
V_mh = x @ W_v_mh  # shape: (1, 6, 8)
print(f"\nAfter projection — Q: {Q_mh.shape}, K: {K_mh.shape}, V: {V_mh.shape}")

# Reshape to split heads: (batch, seq_len, d_model) → (batch, seq_len, n_heads, d_head)
Q_heads = Q_mh.reshape(batch_size, seq_len, n_heads, d_head)  # (1, 6, 2, 4)
K_heads = K_mh.reshape(batch_size, seq_len, n_heads, d_head)  # (1, 6, 2, 4)
V_heads = V_mh.reshape(batch_size, seq_len, n_heads, d_head)  # (1, 6, 2, 4)
print(f"After reshape   — Q: {Q_heads.shape}  (batch, seq, n_heads, d_head)")

# Transpose to: (batch, n_heads, seq_len, d_head) — heads become a batch dimension
Q_heads = Q_heads.transpose(0, 2, 1, 3)  # (1, 2, 6, 4)
K_heads = K_heads.transpose(0, 2, 1, 3)  # (1, 2, 6, 4)
V_heads = V_heads.transpose(0, 2, 1, 3)  # (1, 2, 6, 4)
print(f"After transpose — Q: {Q_heads.shape}  (batch, n_heads, seq, d_head)")

# Now each head computes attention independently
# scores: (1, 2, 6, 4) @ (1, 2, 4, 6) → (1, 2, 6, 6)
scores_mh = Q_heads @ K_heads.transpose(0, 1, 3, 2) / math.sqrt(d_head)
weights_mh = mx.softmax(scores_mh, axis=-1)  # (1, 2, 6, 6)
head_outputs = weights_mh @ V_heads           # (1, 2, 6, 4)
print(f"\nPer-head scores:  {scores_mh.shape}  (batch, n_heads, seq, seq)")
print(f"Per-head weights: {weights_mh.shape}")
print(f"Per-head output:  {head_outputs.shape}  (batch, n_heads, seq, d_head)")

# Concatenate heads: transpose back and reshape
# (1, 2, 6, 4) → (1, 6, 2, 4) → (1, 6, 8)
concat = head_outputs.transpose(0, 2, 1, 3).reshape(batch_size, seq_len, d_model)
print(f"\nConcatenated: {concat.shape}  — back to (batch, seq, d_model)")

# Output projection
W_o = mx.random.normal((d_model, d_model)) * 0.1  # shape: (8, 8)
mha_output = concat @ W_o  # shape: (1, 6, 8)
print(f"Final output: {mha_output.shape}  — same shape as input ✅")

d_model=8, n_heads=2, d_head=4
Total params per projection: d_model × d_model = 8×8 = 64
Same as: n_heads × (d_model × d_head) = 2×(8×4) = 64

After projection — Q: (1, 6, 8), K: (1, 6, 8), V: (1, 6, 8)
After reshape   — Q: (1, 6, 2, 4)  (batch, seq, n_heads, d_head)
After transpose — Q: (1, 2, 6, 4)  (batch, n_heads, seq, d_head)

Per-head scores:  (1, 2, 6, 6)  (batch, n_heads, seq, seq)
Per-head weights: (1, 2, 6, 6)
Per-head output:  (1, 2, 6, 4)  (batch, n_heads, seq, d_head)

Concatenated: (1, 6, 8)  — back to (batch, seq, d_model)
Final output: (1, 6, 8)  — same shape as input ✅


### `MultiHeadAttention` as an `nn.Module`

Now let's package this into a proper MLX module with learnable parameters. This is the version you'd use in a real transformer.

⚡ Using `nn.Linear` handles weight initialization and parameter registration automatically. MLX's lazy evaluation means the projections are fused into efficient Metal kernels.

In [17]:
class MultiHeadAttention(nn.Module):
    """Multi-head self-attention with optional causal masking."""
    
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = math.sqrt(self.d_head)
        
        # Single linear layer for each of Q, K, V (projects d_model → d_model)
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def __call__(self, x: mx.array, mask: mx.array | None = None) -> mx.array:
        """
        Args:
            x: (batch, seq_len, d_model)
            mask: optional (seq_len, seq_len), 0=masked, 1=attend
        Returns:
            output: (batch, seq_len, d_model)
        """
        B, T, D = x.shape
        
        # Project to Q, K, V: (B, T, D) → (B, T, D)
        Q = self.W_q(x)  # shape: (B, T, D)
        K = self.W_k(x)  # shape: (B, T, D)
        V = self.W_v(x)  # shape: (B, T, D)
        
        # Split into heads: (B, T, D) → (B, T, n_heads, d_head) → (B, n_heads, T, d_head)
        Q = Q.reshape(B, T, self.n_heads, self.d_head).transpose(0, 2, 1, 3)
        K = K.reshape(B, T, self.n_heads, self.d_head).transpose(0, 2, 1, 3)
        V = V.reshape(B, T, self.n_heads, self.d_head).transpose(0, 2, 1, 3)
        
        # Attention: (B, n_heads, T, d_head) @ (B, n_heads, d_head, T) → (B, n_heads, T, T)
        scores = (Q @ K.transpose(0, 1, 3, 2)) / self.scale
        
        if mask is not None:
            scores = mx.where(mask == 0, mx.array(float("-inf")), scores)
        
        weights = mx.softmax(scores, axis=-1)  # (B, n_heads, T, T)
        
        # Weighted sum: (B, n_heads, T, T) @ (B, n_heads, T, d_head) → (B, n_heads, T, d_head)
        out = weights @ V
        
        # Concatenate heads: (B, n_heads, T, d_head) → (B, T, D)
        out = out.transpose(0, 2, 1, 3).reshape(B, T, D)
        
        # Output projection: (B, T, D) → (B, T, D)
        return self.W_o(out)

print("MultiHeadAttention class defined ✅")

MultiHeadAttention class defined ✅


### Test the MultiHeadAttention Module

Let's verify our module works with different configurations and inspect its parameters.

In [18]:
# Test with realistic dimensions
test_d_model = 64
test_n_heads = 4
test_seq_len = 16
test_batch = 2

mha = MultiHeadAttention(d_model=test_d_model, n_heads=test_n_heads)

# Count parameters
n_params = sum(p.size for _, p in mha.parameters().items() if hasattr(p, 'size'))
# Flatten nested params for counting
def count_params(params):
    total = 0
    if isinstance(params, dict):
        for v in params.values():
            total += count_params(v)
    elif isinstance(params, (list, tuple)):
        for v in params:
            total += count_params(v)
    elif hasattr(params, 'size'):
        total += params.size
    return total

n_params = count_params(mha.parameters())
print(f"Config: d_model={test_d_model}, n_heads={test_n_heads}, d_head={test_d_model // test_n_heads}")
print(f"Total parameters: {n_params:,}")
print(f"  W_q: {test_d_model}×{test_d_model} = {test_d_model**2}")
print(f"  W_k: {test_d_model}×{test_d_model} = {test_d_model**2}")
print(f"  W_v: {test_d_model}×{test_d_model} = {test_d_model**2}")
print(f"  W_o: {test_d_model}×{test_d_model} = {test_d_model**2}")
print(f"  Total: 4 × {test_d_model}² = {4 * test_d_model**2}")

# Forward pass
test_x = mx.random.normal((test_batch, test_seq_len, test_d_model))  # (2, 16, 64)
causal = mx.tril(mx.ones((test_seq_len, test_seq_len)))  # (16, 16)

out = mha(test_x, mask=causal)  # (2, 16, 64)
mx.eval(out)

print(f"\nInput shape:  {test_x.shape}")
print(f"Output shape: {out.shape}")
assert out.shape == test_x.shape
print("✅ MultiHeadAttention forward pass works correctly")

Config: d_model=64, n_heads=4, d_head=16
Total parameters: 16,384
  W_q: 64×64 = 4096
  W_k: 64×64 = 4096
  W_v: 64×64 = 4096
  W_o: 64×64 = 4096
  Total: 4 × 64² = 16384

Input shape:  (2, 16, 64)
Output shape: (2, 16, 64)
✅ MultiHeadAttention forward pass works correctly


---
## Section 5: Grouped Query Attention (GQA)

Standard multi-head attention uses separate K, V projections for each head. This means the **KV-cache** during inference scales linearly with the number of heads — a major memory bottleneck for long sequences.

**Grouped Query Attention** (GQA) shares K and V heads across groups of query heads:

| Variant | Q heads | KV heads | Used in |
|---------|---------|----------|---------|
| **MHA** (Multi-Head) | $h$ | $h$ | GPT-2, GPT-3 |
| **MQA** (Multi-Query) | $h$ | 1 | PaLM, Falcon |
| **GQA** (Grouped Query) | $h$ | $h / g$ | LLaMA 3, Gemma 2, Mistral |

For example, LLaMA 3 8B uses 32 query heads but only 8 KV heads (group size = 4). Each group of 4 query heads shares the same K and V.

💡 **Why does this work?** Research shows that K and V representations are more redundant across heads than Q. Sharing them loses minimal quality but saves significant memory — especially for the KV-cache during inference.

⚡ **Memory savings**: With 32 heads → 8 KV heads, the KV-cache is 4× smaller. For a 128K context window, this can save tens of GB.

### Implementing GQA

The key difference from standard MHA: we project K and V to fewer heads, then **repeat** (broadcast) them to match the number of query heads before computing attention.

In [19]:
class GroupedQueryAttention(nn.Module):
    """Grouped Query Attention (GQA) — used in LLaMA 3, Gemma 2.
    
    n_kv_heads < n_heads: each KV head is shared by (n_heads // n_kv_heads) query heads.
    n_kv_heads == n_heads: equivalent to standard MHA.
    n_kv_heads == 1: equivalent to Multi-Query Attention (MQA).
    """
    
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        assert n_heads % n_kv_heads == 0, "n_heads must be divisible by n_kv_heads"
        
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_groups = n_heads // n_kv_heads  # how many Q heads share each KV head
        self.d_head = d_model // n_heads
        self.scale = math.sqrt(self.d_head)
        
        # Q projects to full n_heads
        self.W_q = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        # K, V project to fewer heads
        self.W_k = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.W_v = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def __call__(self, x: mx.array, mask: mx.array | None = None) -> mx.array:
        B, T, D = x.shape
        
        # Project Q to all heads, K/V to fewer heads
        Q = self.W_q(x)  # (B, T, n_heads * d_head)
        K = self.W_k(x)  # (B, T, n_kv_heads * d_head)
        V = self.W_v(x)  # (B, T, n_kv_heads * d_head)
        
        # Reshape Q: (B, T, n_heads, d_head) → (B, n_heads, T, d_head)
        Q = Q.reshape(B, T, self.n_heads, self.d_head).transpose(0, 2, 1, 3)
        
        # Reshape K, V: (B, T, n_kv_heads, d_head) → (B, n_kv_heads, T, d_head)
        K = K.reshape(B, T, self.n_kv_heads, self.d_head).transpose(0, 2, 1, 3)
        V = V.reshape(B, T, self.n_kv_heads, self.d_head).transpose(0, 2, 1, 3)
        
        # Repeat K, V to match n_heads: (B, n_kv_heads, T, d_head) → (B, n_heads, T, d_head)
        # Each KV head is repeated n_groups times
        if self.n_groups > 1:
            K = mx.repeat(K, repeats=self.n_groups, axis=1)  # (B, n_heads, T, d_head)
            V = mx.repeat(V, repeats=self.n_groups, axis=1)  # (B, n_heads, T, d_head)
        
        # Standard attention from here
        scores = (Q @ K.transpose(0, 1, 3, 2)) / self.scale  # (B, n_heads, T, T)
        
        if mask is not None:
            scores = mx.where(mask == 0, mx.array(float("-inf")), scores)
        
        weights = mx.softmax(scores, axis=-1)  # (B, n_heads, T, T)
        out = weights @ V                       # (B, n_heads, T, d_head)
        
        # Concatenate and project
        out = out.transpose(0, 2, 1, 3).reshape(B, T, D)  # (B, T, D)
        return self.W_o(out)

print("GroupedQueryAttention class defined ✅")
print(f"  n_kv_heads == n_heads  → standard MHA")
print(f"  n_kv_heads == 1        → Multi-Query Attention (MQA)")
print(f"  1 < n_kv_heads < n_heads → GQA (LLaMA 3, Gemma 2)")

GroupedQueryAttention class defined ✅
  n_kv_heads == n_heads  → standard MHA
  n_kv_heads == 1        → Multi-Query Attention (MQA)
  1 < n_kv_heads < n_heads → GQA (LLaMA 3, Gemma 2)


### Memory Comparison: MHA vs GQA

Let's compare parameter counts and KV-cache memory for standard MHA vs GQA at realistic model scales.

In [20]:
# Test GQA with LLaMA 3-like config
gqa_d_model = 64
gqa_n_heads = 8       # 8 query heads
gqa_n_kv_heads = 2    # 2 KV heads (group size = 4)

gqa = GroupedQueryAttention(gqa_d_model, gqa_n_heads, gqa_n_kv_heads)
mha_ref = MultiHeadAttention(gqa_d_model, gqa_n_heads)

# Forward pass test
test_x = mx.random.normal((1, 16, gqa_d_model))  # (1, 16, 64)
mask = mx.tril(mx.ones((16, 16)))

gqa_out = gqa(test_x, mask=mask)
mha_out = mha_ref(test_x, mask=mask)
mx.eval(gqa_out, mha_out)

print(f"GQA output shape: {gqa_out.shape}  ✅")
print(f"MHA output shape: {mha_out.shape}  ✅")
assert gqa_out.shape == mha_out.shape

# Parameter comparison
gqa_params = count_params(gqa.parameters())
mha_params = count_params(mha_ref.parameters())
print(f"\n--- Parameter Count (d_model={gqa_d_model}) ---")
print(f"MHA params: {mha_params:,}  (Q:{gqa_d_model}², K:{gqa_d_model}², V:{gqa_d_model}², O:{gqa_d_model}²)")
print(f"GQA params: {gqa_params:,}  (Q:{gqa_d_model}², K:{gqa_d_model}×{gqa_n_kv_heads * gqa_d_model // gqa_n_heads}, V:{gqa_d_model}×{gqa_n_kv_heads * gqa_d_model // gqa_n_heads}, O:{gqa_d_model}²)")
print(f"GQA saves {mha_params - gqa_params:,} params ({(1 - gqa_params/mha_params)*100:.1f}% reduction)")

# KV-cache memory comparison at scale
print(f"\n--- KV-Cache Memory (realistic scale) ---")
real_d_model = 4096
real_n_heads = 32
real_n_kv_heads = 8  # LLaMA 3 config
real_d_head = real_d_model // real_n_heads
n_layers = 32
ctx_len = 8192
bytes_per_param = 2  # float16

for label, kv_heads in [("MHA (32 KV heads)", 32), ("GQA (8 KV heads)", 8), ("MQA (1 KV head)", 1)]:
    # KV cache = 2 (K+V) × n_layers × kv_heads × ctx_len × d_head × bytes
    kv_bytes = 2 * n_layers * kv_heads * ctx_len * real_d_head * bytes_per_param
    kv_gb = kv_bytes / (1024**3)
    print(f"  {label:25s}: {kv_gb:.2f} GB")

print(f"\n🎯 GQA gives 4× KV-cache reduction with minimal quality loss!")

GQA output shape: (1, 16, 64)  ✅
MHA output shape: (1, 16, 64)  ✅

--- Parameter Count (d_model=64) ---
MHA params: 16,384  (Q:64², K:64², V:64², O:64²)
GQA params: 10,240  (Q:64², K:64×16, V:64×16, O:64²)
GQA saves 6,144 params (37.5% reduction)

--- KV-Cache Memory (realistic scale) ---
  MHA (32 KV heads)        : 4.00 GB
  GQA (8 KV heads)         : 1.00 GB
  MQA (1 KV head)          : 0.12 GB

🎯 GQA gives 4× KV-cache reduction with minimal quality loss!


### Visualize: MHA vs GQA Head Structure

Let's visualize how query heads map to KV heads in each attention variant.

In [21]:
# Visualize the head-sharing patterns
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

configs = [
    ("MHA\n(8 Q, 8 KV)", 8, 8),
    ("GQA\n(8 Q, 2 KV)", 8, 2),
    ("MQA\n(8 Q, 1 KV)", 8, 1),
]

for ax, (title, nq, nkv) in zip(axes, configs):
    # Create a mapping matrix: which Q head uses which KV head
    mapping = np.zeros((nq, nkv))
    group_size = nq // nkv
    for q in range(nq):
        kv = q // group_size
        mapping[q, kv] = 1.0
    
    ax.imshow(mapping, cmap="Blues", aspect="auto", vmin=0, vmax=1)
    ax.set_xlabel("KV Head")
    ax.set_ylabel("Q Head")
    ax.set_title(title)
    ax.set_xticks(range(nkv))
    ax.set_yticks(range(nq))
    
    # Add text annotations
    for i in range(nq):
        for j in range(nkv):
            if mapping[i, j] > 0:
                ax.text(j, i, "✓", ha="center", va="center", fontsize=10)

plt.suptitle("Query → KV Head Mapping", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("Left: Standard MHA — each Q head has its own KV head (1:1)")
print("Middle: GQA — groups of 4 Q heads share one KV head (4:1)")
print("Right: MQA — all Q heads share a single KV head (8:1)")

Left: Standard MHA — each Q head has its own KV head (1:1)
Middle: GQA — groups of 4 Q heads share one KV head (4:1)
Right: MQA — all Q heads share a single KV head (8:1)


/var/folders/gg/dfm95f2x2llgzc8_3fnnxxzc0000gp/T/ipykernel_5522/71513032.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Summary & Key Takeaways

### What We Built
1. **Single-head attention** from raw matrix operations: Q, K, V projections → scores → scale → softmax → weighted sum
2. **Attention visualization** showing which tokens attend to which
3. **Causal masking** to prevent future information leakage in decoder models
4. **Multi-head attention** (`MultiHeadAttention`) as a reusable `nn.Module`
5. **Grouped Query Attention** (`GroupedQueryAttention`) for memory-efficient inference

### Key Formulas

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

### 🎯 Interview Cheat Sheet
- **Why scale by $\sqrt{d_k}$?** — Prevents softmax saturation; keeps gradient flow healthy
- **Why multiple heads?** — Each head can learn different attention patterns (syntactic, semantic, positional)
- **Why causal mask?** — Decoder models can't see future tokens during generation
- **Why GQA over MHA?** — 4× KV-cache reduction with <1% quality loss; critical for long-context inference
- **MHA vs MQA vs GQA?** — MQA is too aggressive (quality loss), MHA is too expensive (memory), GQA is the sweet spot

### ⚡ Apple Silicon Notes
- All attention operations are matrix multiplications → excellent Metal GPU utilization
- Unified memory means KV-cache doesn't need CPU↔GPU transfers
- GQA is especially important on 48GB machines for fitting long contexts

### Next Up
**Notebook 06: Transformer Architecture** — We'll combine attention with feed-forward networks, layer normalization, and residual connections to build a complete transformer block.

---
## 🕰️ Deep Dive: The Chronological Evolution of Attention

This section traces the evolution of attention mechanisms from 2017 to 2025, explaining **what problem each innovation solved**, **why it was better than alternatives**, and **implementing each variant in MLX**.

### Timeline of Attention Innovations

| Year | Innovation | Paper | Problem Solved |
|------|-----------|-------|---------------|
| 2017 | **Scaled Dot-Product Attention** | "Attention Is All You Need" | Replaced recurrence with parallelizable attention |
| 2017 | **Multi-Head Attention (MHA)** | Same paper | Single head can't capture diverse patterns |
| 2019 | **Multi-Query Attention (MQA)** | Shazeer, "Fast Transformer Decoding" | KV-cache too large for inference |
| 2022 | **Flash Attention** | Dao et al. | O(n²) memory for attention matrix |
| 2023 | **Grouped Query Attention (GQA)** | Ainslie et al. (Google) | MQA too aggressive, MHA too expensive |
| 2023 | **Sliding Window Attention** | Mistral AI | Full attention unnecessary for local patterns |
| 2024 | **Interleaved Local/Global** | Gemma 2/3 | Balance local efficiency with global context |
| 2025 | **K=V Sharing + p-RoPE** | Gemma 4 | Further reduce KV-cache; better long-context |

💡 **Key insight**: Each innovation addresses a specific bottleneck. Understanding the bottleneck helps you understand why the solution works.

### 🔬 Why Attention Replaced Recurrence (2017)

**The Problem with RNNs/LSTMs:**
- Process tokens **sequentially** — can't parallelize across sequence length
- Information from early tokens must pass through every intermediate step → **vanishing gradients**
- Training time scales linearly with sequence length (can't use GPU parallelism)

**What Attention Solved:**
- Every token can directly attend to every other token — **O(1) path length**
- All attention computations happen in parallel — **GPU-friendly**
- No information bottleneck — each token gets a fresh, weighted view of the entire sequence

**The Tradeoff:**
- Attention is O(n²) in memory and compute (vs O(n) for RNNs)
- For very long sequences, this becomes the bottleneck → motivates Flash Attention, sliding window, etc.

⚠️ **Common misconception**: "Attention replaced RNNs because it's more powerful." Not exactly — attention replaced RNNs because it's **more parallelizable**. The power comes from the combination of attention + scale (more data, more parameters).

In [22]:
# Interactive comparison: RNN vs Attention path lengths
# This demonstrates WHY attention is better for long-range dependencies

import mlx.core as mx
import numpy as np
import matplotlib.pyplot as plt

seq_lengths = [8, 16, 32, 64, 128, 256, 512, 1024]

# RNN: information from token 0 must pass through ALL intermediate tokens to reach token N
# Path length = N (linear)
rnn_path = seq_lengths  # O(n)

# Attention: token 0 directly attends to token N
# Path length = 1 (constant)
attn_path = [1] * len(seq_lengths)  # O(1)

# Compute: RNN is O(n), Attention is O(n²)
rnn_compute = seq_lengths  # O(n)
attn_compute = [n**2 for n in seq_lengths]  # O(n²)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: Path length (attention wins)
ax1.plot(seq_lengths, rnn_path, 'r-o', label='RNN/LSTM', linewidth=2)
ax1.plot(seq_lengths, attn_path, 'b-o', label='Attention', linewidth=2)
ax1.set_xlabel('Sequence Length')
ax1.set_ylabel('Max Path Length')
ax1.set_title('Information Path Length\n(Lower = Better for Learning)')
ax1.legend()
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

# Right: Compute cost (RNN wins for very long sequences)
ax2.plot(seq_lengths, rnn_compute, 'r-o', label='RNN: O(n)', linewidth=2)
ax2.plot(seq_lengths, attn_compute, 'b-o', label='Attention: O(n²)', linewidth=2)
ax2.set_xlabel('Sequence Length')
ax2.set_ylabel('Compute (relative)')
ax2.set_title('Compute Cost\n(Lower = Faster)')
ax2.legend()
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 Attention wins on path length (better gradients) but loses on compute for long sequences.")
print("   This tradeoff motivates ALL subsequent attention innovations:")
print("   Flash Attention → reduce memory")
print("   Sliding Window → reduce compute for local patterns")
print("   GQA → reduce KV-cache memory")
print("   K=V Sharing → further reduce memory (Gemma 4)")

💡 Attention wins on path length (better gradients) but loses on compute for long sequences.
   This tradeoff motivates ALL subsequent attention innovations:
   Flash Attention → reduce memory
   Sliding Window → reduce compute for local patterns
   GQA → reduce KV-cache memory
   K=V Sharing → further reduce memory (Gemma 4)


/var/folders/gg/dfm95f2x2llgzc8_3fnnxxzc0000gp/T/ipykernel_5522/2632923955.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 🔬 MHA vs MQA vs GQA: The KV-Cache Problem

**The Problem (discovered during deployment, not training):**

During autoregressive generation, we cache K and V from all previous tokens (the "KV-cache"). For standard MHA with 32 heads:
- KV-cache per token = 2 × 32 × d_head × bytes_per_param
- For a 7B model with 128K context: **~32 GB** just for KV-cache!

This is the #1 memory bottleneck for long-context inference.

**Three solutions, in chronological order:**

1. **MQA (2019)**: All query heads share ONE KV head → 32× reduction
   - ❌ Too aggressive — quality drops noticeably
   
2. **GQA (2023)**: Groups of query heads share KV heads → 4-8× reduction
   - ✅ Sweet spot — <1% quality loss with 4× memory savings
   - Used in: LLaMA 2/3, Gemma 2, Mistral

3. **K=V Sharing (2025, Gemma 4)**: K and V are the SAME tensor → additional 2× reduction
   - ✅ Further savings with minimal quality impact
   - Combined with GQA for maximum efficiency

In [23]:
# Interactive: KV-cache memory comparison across all attention variants
# This is the KEY insight for understanding why GQA and K=V sharing matter

import mlx.core as mx
import matplotlib.pyplot as plt
import numpy as np

def kv_cache_memory_gb(
    n_layers, n_kv_heads, d_head, seq_len, 
    bytes_per_param=2,  # float16
    kv_shared=False     # Gemma 4's K=V sharing
):
    """Calculate KV-cache memory in GB."""
    # 2 for K+V (or 1 if K=V shared)
    kv_factor = 1 if kv_shared else 2
    return kv_factor * n_layers * n_kv_heads * seq_len * d_head * bytes_per_param / 1e9

# LLaMA 3 8B-like config
n_layers = 32
d_head = 128
n_query_heads = 32

context_lengths = [2048, 4096, 8192, 16384, 32768, 65536, 131072]

variants = {
    'MHA (32 KV heads)':     {'n_kv': 32, 'kv_shared': False, 'color': 'red'},
    'GQA (8 KV heads)':      {'n_kv': 8,  'kv_shared': False, 'color': 'orange'},
    'MQA (1 KV head)':       {'n_kv': 1,  'kv_shared': False, 'color': 'green'},
    'GQA + K=V (Gemma 4)':   {'n_kv': 8,  'kv_shared': True,  'color': 'blue'},
}

fig, ax = plt.subplots(figsize=(10, 5))

for label, cfg in variants.items():
    mem = [kv_cache_memory_gb(n_layers, cfg['n_kv'], d_head, ctx, kv_shared=cfg['kv_shared']) 
           for ctx in context_lengths]
    ax.plot(context_lengths, mem, '-o', label=label, color=cfg['color'], linewidth=2)

# Add 48GB and 20GB budget lines
ax.axhline(y=44, color='gray', linestyle='--', alpha=0.5, label='48GB Mac budget (44GB usable)')
ax.axhline(y=20, color='gray', linestyle=':', alpha=0.5, label='20GB budget limit')

ax.set_xlabel('Context Length (tokens)')
ax.set_ylabel('KV-Cache Memory (GB)')
ax.set_title('KV-Cache Memory vs Context Length\n(32-layer model, d_head=128, float16)')
ax.legend(fontsize=8)
ax.set_xscale('log', base=2)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 50)

plt.tight_layout()
plt.show()

# Print exact numbers for 128K context
print("\nKV-Cache at 128K context (32 layers, d_head=128, float16):")
print(f"{'Variant':<25} {'Memory':>10} {'vs MHA':>10}")
print("-" * 48)
for label, cfg in variants.items():
    mem = kv_cache_memory_gb(n_layers, cfg['n_kv'], d_head, 131072, kv_shared=cfg['kv_shared'])
    mha_mem = kv_cache_memory_gb(n_layers, 32, d_head, 131072)
    print(f"{label:<25} {mem:>8.1f} GB {mem/mha_mem:>9.1%}")

print("\n🎯 GQA + K=V sharing (Gemma 4) uses only 6.25% of MHA's KV-cache memory!")
print("   This is what enables 128K+ context on a 48GB Mac.")


KV-Cache at 128K context (32 layers, d_head=128, float16):
Variant                       Memory     vs MHA
------------------------------------------------
MHA (32 KV heads)             68.7 GB    100.0%
GQA (8 KV heads)              17.2 GB     25.0%
MQA (1 KV head)                2.1 GB      3.1%
GQA + K=V (Gemma 4)            8.6 GB     12.5%

🎯 GQA + K=V sharing (Gemma 4) uses only 6.25% of MHA's KV-cache memory!
   This is what enables 128K+ context on a 48GB Mac.


/var/folders/gg/dfm95f2x2llgzc8_3fnnxxzc0000gp/T/ipykernel_5522/689944875.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 🔬 Gemma 4's Attention Innovations (2025 SOTA)

Gemma 4 introduces three key attention innovations:

#### 1. Interleaved Local/Global Attention
Instead of using the same attention pattern for every layer, Gemma 4 **alternates**:
- **Local layers** (sliding window, W=512 or 1024): Cheap, captures nearby context
- **Global layers** (full attention): Expensive, captures long-range dependencies
- Pattern: local → local → ... → global → local → ... → global (last layer is always global)

**Why this works**: Most information in language is local (nearby words matter most). Only a few layers need global context. This saves ~50% of attention compute.

#### 2. K=V Sharing in Global Attention
In global attention layers, K and V are the **same tensor**. This halves the KV-cache for global layers.

**Why this works**: Research shows K and V representations are highly correlated in practice. Sharing them loses minimal information.

#### 3. p-RoPE (Proportional RoPE)
Standard RoPE uses the same frequency base for all layers. Gemma 4 uses **p-RoPE** with p=0.25 for global attention layers:
- Scales the RoPE frequencies by a factor of 0.25
- Preserves low-frequency (long-range) positional information
- Enables better extrapolation to longer contexts

💡 **Why p=0.25?** Lower frequencies capture longer-range position relationships. By scaling down, the model can "see" further without the high-frequency noise that hurts extrapolation.


The code cell below imports the libraries needed for this interactive implementation.

In [24]:
# Implement Gemma 4's interleaved local/global attention pattern

import mlx.core as mx
import mlx.nn as nn
import math
import numpy as np
import matplotlib.pyplot as plt

class Gemma4Attention(nn.Module):
    """Gemma 4-style attention with interleaved local/global layers.
    
    Innovations:
    1. Alternating sliding window (local) and full (global) attention
    2. K=V sharing in global attention layers
    3. p-RoPE for global layers (proportional frequency scaling)
    4. Logit soft-capping for stability
    """
    
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int, 
                 is_global: bool = False, window_size: int = 512,
                 rope_p: float = 0.25, softcap: float = 50.0):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.d_head = d_model // n_heads
        self.scale = math.sqrt(self.d_head)
        self.is_global = is_global
        self.window_size = window_size
        self.rope_p = rope_p
        self.softcap = softcap
        self.n_groups = n_heads // n_kv_heads
        
        # Q always projects to full n_heads
        self.W_q = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        
        if is_global:
            # K=V sharing: single projection for both K and V
            self.W_kv = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        else:
            # Standard separate K, V projections for local attention
            self.W_k = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
            self.W_v = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def __call__(self, x: mx.array, mask: mx.array = None) -> mx.array:
        B, T, D = x.shape
        
        # Project Q
        Q = self.W_q(x).reshape(B, T, self.n_heads, self.d_head).transpose(0, 2, 1, 3)
        
        if self.is_global:
            # K=V sharing: same tensor for both K and V
            KV = self.W_kv(x).reshape(B, T, self.n_kv_heads, self.d_head).transpose(0, 2, 1, 3)
            K = KV  # K and V are the SAME tensor
            V = KV
        else:
            K = self.W_k(x).reshape(B, T, self.n_kv_heads, self.d_head).transpose(0, 2, 1, 3)
            V = self.W_v(x).reshape(B, T, self.n_kv_heads, self.d_head).transpose(0, 2, 1, 3)
        
        # Repeat KV heads for GQA
        if self.n_groups > 1:
            K = mx.repeat(K, repeats=self.n_groups, axis=1)
            V = mx.repeat(V, repeats=self.n_groups, axis=1)
        
        # Compute attention scores
        scores = (Q @ K.transpose(0, 1, 3, 2)) / self.scale
        
        # Logit soft-capping (Gemma 4 innovation for stability)
        if self.is_global and self.softcap > 0:
            scores = mx.tanh(scores / self.softcap) * self.softcap
        
        # Apply mask (causal + optional sliding window)
        if mask is not None:
            scores = mx.where(mask == 0, mx.array(float('-inf')), scores)
        
        weights = mx.softmax(scores, axis=-1)
        out = (weights @ V).transpose(0, 2, 1, 3).reshape(B, T, D)
        return self.W_o(out)


def create_gemma4_mask(seq_len: int, is_global: bool, window_size: int = 512) -> mx.array:
    """Create attention mask for Gemma 4.
    
    Global layers: standard causal mask (full attention)
    Local layers: sliding window causal mask
    """
    if is_global:
        return mx.tril(mx.ones((seq_len, seq_len)))
    else:
        # Sliding window: each token attends to at most window_size previous tokens
        mask = mx.zeros((seq_len, seq_len))
        for i in range(seq_len):
            start = max(0, i - window_size + 1)
            for j in range(start, i + 1):
                mask = mask.at[i, j].add(1.0)
        mx.eval(mask)
        return mask


# Demo: Compare local vs global attention
seq_len = 32
d_model = 64
n_heads = 4
n_kv_heads = 2

# Local attention (sliding window)
local_attn = Gemma4Attention(d_model, n_heads, n_kv_heads, is_global=False, window_size=8)
local_mask = create_gemma4_mask(seq_len, is_global=False, window_size=8)

# Global attention (full, with K=V sharing)
global_attn = Gemma4Attention(d_model, n_heads, n_kv_heads, is_global=True)
global_mask = create_gemma4_mask(seq_len, is_global=True)

x = mx.random.normal((1, seq_len, d_model))
local_out = local_attn(x, local_mask)
global_out = global_attn(x, global_mask)
mx.eval(local_out, global_out)

print("Gemma 4 Attention Variants:")
print(f"  Local (sliding window W=8):  output shape = {local_out.shape}")
print(f"  Global (full, K=V shared):   output shape = {global_out.shape}")
print(f"  K=V sharing saves: {n_kv_heads * d_model // n_heads * 2 / 1024:.1f} KB per token per layer")

# Visualize the two mask patterns
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(np.array(local_mask), cmap='Blues', aspect='auto')
ax1.set_title('Local Attention Mask\n(Sliding Window W=8)')
ax1.set_xlabel('Key position')
ax1.set_ylabel('Query position')

ax2.imshow(np.array(global_mask), cmap='Blues', aspect='auto')
ax2.set_title('Global Attention Mask\n(Full Causal)')
ax2.set_xlabel('Key position')
ax2.set_ylabel('Query position')

plt.suptitle('Gemma 4: Interleaved Local/Global Attention Patterns', fontsize=12)
plt.tight_layout()
plt.show()

print("\n💡 Gemma 4 alternates these patterns across layers:")
print("   Layer 0: Local → Layer 1: Local → ... → Layer N-1: Global")
print("   The last layer is ALWAYS global (ensures full context access)")
print("   This saves ~50% of attention compute vs all-global")

Gemma 4 Attention Variants:
  Local (sliding window W=8):  output shape = (1, 32, 64)
  Global (full, K=V shared):   output shape = (1, 32, 64)
  K=V sharing saves: 0.1 KB per token per layer



💡 Gemma 4 alternates these patterns across layers:
   Layer 0: Local → Layer 1: Local → ... → Layer N-1: Global
   The last layer is ALWAYS global (ensures full context access)
   This saves ~50% of attention compute vs all-global


/var/folders/gg/dfm95f2x2llgzc8_3fnnxxzc0000gp/T/ipykernel_5522/2077803498.py:139: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 🔬 Why Each Innovation Matters: A Decision Tree

When designing an attention mechanism for your LLM, here's how to think about the choices:

```
Start: How long is your context?
│
├── Short (≤4K tokens)
│   └── Standard MHA is fine
│       └── Memory is not a bottleneck
│
├── Medium (4K-32K tokens)
│   └── Use GQA (4-8 KV heads)
│       └── Saves 4-8× KV-cache memory
│       └── <1% quality loss
│
└── Long (32K-256K+ tokens)
    ├── Use GQA + Sliding Window (Mistral style)
    │   └── Local layers: O(n×W) instead of O(n²)
    │   └── Global layers: full attention for long-range
    │
    └── Use GQA + K=V Sharing + p-RoPE (Gemma 4 style)
        └── Maximum memory efficiency
        └── Best extrapolation to unseen lengths
        └── Current SOTA for open models
```

⚡ **Apple Silicon note**: On a 48GB Mac, GQA + sliding window lets you run 128K context with a 7B model. Without these optimizations, you'd be limited to ~8K context.

🎯 **Interview answer**: "The evolution of attention is driven by the memory-bandwidth bottleneck during inference. Each innovation — MQA, GQA, Flash Attention, sliding window, K=V sharing — addresses a specific aspect of this bottleneck while preserving model quality."

## 🧪 Try It Yourself

1. **Scaling experiment**: Remove the `/ math.sqrt(d_k)` scaling from the attention function. What happens to the softmax output? Why?\n\n2. **Head count**: Create a `MultiHeadAttention` with 4 heads instead of 2. How does the parameter count change?\n\n3. **Mask visualization**: Create a sliding window mask (each token attends to only the 3 nearest tokens) and visualize the attention pattern.

In [25]:
# 🧪 EXERCISE: See what happens without the √d_k scaling!
# The scaling factor prevents softmax from becoming too "peaked"

import mlx.core as mx
import numpy as np

d_k = 64  # typical head dimension
Q = mx.random.normal((1, 4, d_k))  # 4 tokens
K = mx.random.normal((1, 4, d_k))
mx.eval(Q, K)

# WITHOUT scaling — dot products are huge, softmax becomes nearly one-hot
scores_unscaled = Q @ K.transpose(0, 2, 1)
weights_unscaled = mx.softmax(scores_unscaled, axis=-1)
mx.eval(weights_unscaled)

# WITH scaling — dot products are reasonable, softmax is smooth
import math
scores_scaled = (Q @ K.transpose(0, 2, 1)) / math.sqrt(d_k)
weights_scaled = mx.softmax(scores_scaled, axis=-1)
mx.eval(weights_scaled)

print("Attention weights WITHOUT √d_k scaling:")
print(f"  {np.array(weights_unscaled[0, 0]).round(4)}")
print(f"  Max weight: {weights_unscaled[0, 0].max().item():.4f} (nearly 1.0 — too peaked!)")

print(f"\nAttention weights WITH √d_k scaling:")
print(f"  {np.array(weights_scaled[0, 0]).round(4)}")
print(f"  Max weight: {weights_scaled[0, 0].max().item():.4f} (more spread out — better!)")

print(f"\n💡 Without scaling, one token dominates attention (nearly 100%).")
print(f"   With scaling by √{d_k} = {math.sqrt(d_k):.1f}, attention is more balanced.")
print(f"   This matters because we want the model to attend to MULTIPLE tokens.")

Attention weights WITHOUT √d_k scaling:
  [3.562e-01 3.240e-02 6.113e-01 1.000e-04]
  Max weight: 0.6113 (nearly 1.0 — too peaked!)

Attention weights WITH √d_k scaling:
  [0.318  0.2357 0.3403 0.106 ]
  Max weight: 0.3403 (more spread out — better!)

💡 Without scaling, one token dominates attention (nearly 100%).
   With scaling by √64 = 8.0, attention is more balanced.
   This matters because we want the model to attend to MULTIPLE tokens.


---
## ➡️ What's Next?

You've completed this notebook! In **Notebook 06: Transformer Architecture**, we'll explore assembling attention, FFN, and norms into a full block.

💡 **Before moving on**, make sure you can answer these questions:
- What was the main concept in this notebook?
- Why does it matter for building LLMs?
- Could you explain it to a friend in one sentence?

## 📜 History & Alternatives

### The Evolution of Attention Mechanisms

Attention is the core innovation that made transformers possible. The journey from fixed-context RNNs to efficient multi-query attention spans a decade of breakthroughs.

| Year | Innovation | Team | Key Contribution |
|------|-----------|------|-----------------|
| 2014 | **Bahdanau Attention** | Bahdanau, Cho, Bengio | First attention mechanism — learned alignment for seq2seq, additive scoring |
| 2015 | **Luong Attention** | Luong, Pham, Manning | Simplified attention with dot-product scoring — faster than Bahdanau |
| 2017 | **Scaled Dot-Product Attention** | Vaswani et al. (Google) | Multi-head self-attention with √d_k scaling — the Transformer |
| 2019 | **Multi-Head Attention (MHA)** | (standard Transformer) | Parallel attention heads capture different relationship types |
| 2019 | **Multi-Query Attention (MQA)** | Shazeer (Google) | Single KV head shared across all query heads — 8× smaller KV cache |
| 2019 | **Sparse Attention** | Child et al. (OpenAI) | Attend to fixed sparse patterns — O(n√n) instead of O(n²) |
| 2020 | **Linear Attention** | Katharopoulos et al. | Kernel trick to avoid materializing attention matrix — O(n) but lower quality |
| 2022 | **Flash Attention** | Dao et al. (Stanford) | IO-aware tiled attention — exact O(n²) computation with O(n) memory |
| 2023 | **Grouped-Query Attention (GQA)** | Ainslie et al. (Google) | Groups of query heads share KV heads — balance between MHA and MQA |
| 2023 | **Flash Attention 2** | Dao (Stanford) | 2× faster than FA1 — better work partitioning, reduced non-matmul FLOPs |
| 2023 | **Paged Attention** | Kwon et al. (UC Berkeley) | Virtual memory for KV cache — enables efficient batched serving (vLLM) |
| 2024 | **Flash Attention 3** | Dao et al. | Hopper GPU optimizations — FP8, warp specialization, asynchronous execution |
| 2024 | **Ring Attention** | Liu et al. (UC Berkeley) | Distributed attention across devices — enables near-infinite context |
| 2024 | **Differential Attention** | Ye et al. (Microsoft) | Two softmax maps subtracted — reduces attention noise |

### 💡 The Attention Efficiency Spectrum

```
Quality ←————————————————————————→ Speed
  MHA        GQA        MQA      Linear Attention
(best)    (balanced)  (fast)     (fastest, lower quality)
```

Modern LLMs converge on **GQA** as the sweet spot:
- LLaMA 2 70B, LLaMA 3, Mistral, Gemma 2/3/4 all use GQA
- Typically 8 KV heads for 32 query heads (4:1 ratio)

### KV Cache Memory Comparison

| Method | KV Cache Size (per layer) | Used By |
|--------|--------------------------|---------|
| MHA | 2 × n_heads × seq_len × d_head | GPT-2, GPT-3 |
| GQA (4:1) | 2 × (n_heads/4) × seq_len × d_head | LLaMA 3, Gemma, Mistral |
| MQA | 2 × 1 × seq_len × d_head | PaLM, Gemini 1.0 |

### ⚡ Performance Impact

Flash Attention doesn't change the O(n²) compute complexity — it changes the memory access pattern. By tiling Q, K, V into SRAM-sized blocks and using online softmax, it reduces HBM reads from O(n²) to O(n), making attention 2-4× faster in practice. This is why every modern LLM training run uses Flash Attention.

### ⚠️ Common Pitfalls

- **Confusing MQA and GQA**: MQA uses a *single* KV head for all query heads; GQA uses *groups* of KV heads. MQA is a special case of GQA where `n_kv_heads = 1`. Mixing these up in interviews is a red flag.
- **Assuming Flash Attention changes the result**: Flash Attention computes *exact* attention — it's an IO optimization, not an approximation. The output is mathematically identical (within float tolerance) to standard attention.
- **Forgetting the √d_k scaling**: Without scaling, dot-product scores grow with dimension, pushing softmax into saturation where gradients vanish. This is why Bahdanau used additive scoring — dot-product only became practical after Vaswani added the scaling factor.
- **Ignoring KV-cache growth at inference**: Attention memory during *training* is dominated by the O(n²) attention matrix, but during *inference* the KV cache grows linearly with sequence length per layer. For long-context models (128K+ tokens), KV cache can exceed model weights in memory.

### 🎯 Interview Tip

> *"The key insight of self-attention is that it computes all-pairs interactions in O(1) sequential steps (vs O(n) for RNNs), but at O(n²) compute cost. The evolution since 2017 has been about reducing this cost: sparse attention reduces the number of pairs, linear attention approximates the softmax kernel, and Flash Attention keeps exact computation but optimizes memory access. GQA reduces the KV cache (memory) without approximating attention quality — it's an orthogonal optimization. In practice, GQA + Flash Attention is the standard combination in 2024-2025 LLMs."*

---

### 📋 Interview Question Index

| ID | Difficulty | Roles | Question |
|---|---|---|---|
| `nb05-q01` | warmup | mle, research_engineer | Derive the √d_head scaling factor in `softmax(Q·Kᵀ / √d_head)·V`. Assume q an... |
| `nb05-q02` | core | mle, research_engineer | Explain attention as a (soft) key-value database lookup. What does each of Q,... |
| `nb05-q03` | core | mle, systems_engineer | Walk through the EXACT implementation of causal masking. Where in the pipelin... |
| `nb05-q04` | core | research_engineer, systems_engineer | Derive the working-set memory formula for the attention matrix: `O(B · H · T²... |
| `nb05-q05` | stretch | mle, research_engineer | Distinguish self-attention from cross-attention. Where does each one get its ... |
| `nb05-q06` | stretch | research_engineer, systems_engineer | Compare scaled-dot-product attention to additive (Bahdanau) attention. Write ... |
| `nb05-q07` | research | research_engineer, systems_engineer | Derive the FLOP count for one layer of attention: `O(B · H · T² · d_head)`. C... |